# DATASET ACQUISITION AND INITIAL INSPECTION

In [2]:
# ============================================================
# STEP 1: DATASET ACQUISITION AND INITIAL INSPECTION
# Manuscript: Digital Economy-Driven Prediction of
# Manufacturing Green Total Factor Productivity Using SWO-STGCN
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------------------- 1. File Paths --------------------------

DATASET_PATH = r"G:\5287\digital_economy_gtfp_china.csv"
OUTPUT_DIR = r"G:\5287\Step1_Dataset_Inspection"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------- 2. Load Dataset ------------------------

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PATH}\n"
        "Check the drive, folder, and filename."
    )

df = pd.read_csv(DATASET_PATH, low_memory=False)

print("=" * 65)
print("STEP 1: DATASET ACQUISITION AND INITIAL INSPECTION")
print("=" * 65)

print("\nDataset loaded successfully.")
print(f"Dataset path: {DATASET_PATH}")
print(f"Number of observations: {df.shape[0]}")
print(f"Number of attributes: {df.shape[1]}")

# -------------------- 3. Preview Dataset ---------------------

print("\nFirst 10 Records:")
print(df.head(10))

print("\nLast 5 Records:")
print(df.tail())

# -------------------- 4. Column Information -----------------

column_info = pd.DataFrame({
    "Column_Name": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notna().sum().values,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage": (
        df.isna().mean().values * 100
    ).round(2),
    "Unique_Values": df.nunique(dropna=True).values
})

column_info.to_csv(
    os.path.join(OUTPUT_DIR, "column_information.csv"),
    index=False
)

print("\nColumn Information:")
print(column_info.to_string(index=False))

# -------------------- 5. Missing-Value Analysis --------------

missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isnull().sum().values,
    "Missing_Percentage": (
        df.isnull().mean().values * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "Missing_Count", ascending=False
)

missing_summary.to_csv(
    os.path.join(OUTPUT_DIR, "missing_value_summary.csv"),
    index=False
)

print("\nMissing-Value Summary:")
print(missing_summary.to_string(index=False))

# Missing-value visualization
missing_plot = missing_summary[
    missing_summary["Missing_Count"] > 0
]

if not missing_plot.empty:
    plt.figure(figsize=(12, 6))
    plt.bar(
        missing_plot["Column"],
        missing_plot["Missing_Percentage"]
    )
    plt.xlabel("Dataset Attributes")
    plt.ylabel("Missing Values (%)")
    plt.title("Missing-Value Distribution Across Attributes")
    plt.xticks(rotation=75, ha="right")
    plt.tight_layout()

    plt.savefig(
        os.path.join(OUTPUT_DIR, "missing_value_distribution.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

# -------------------- 6. Duplicate-Record Analysis -----------

duplicate_count = int(df.duplicated().sum())

duplicate_summary = pd.DataFrame({
    "Total_Records": [len(df)],
    "Duplicate_Records": [duplicate_count],
    "Duplicate_Percentage": [
        round(duplicate_count / len(df) * 100, 2)
        if len(df) > 0 else 0
    ]
})

duplicate_summary.to_csv(
    os.path.join(OUTPUT_DIR, "duplicate_record_summary.csv"),
    index=False
)

print("\nDuplicate Records:", duplicate_count)

# -------------------- 7. Numerical Statistical Summary -------

numeric_df = df.select_dtypes(include=np.number)

if not numeric_df.empty:
    descriptive_stats = numeric_df.describe().T
    descriptive_stats["Median"] = numeric_df.median()
    descriptive_stats["Skewness"] = numeric_df.skew()
    descriptive_stats["Kurtosis"] = numeric_df.kurtosis()

    descriptive_stats.to_csv(
        os.path.join(OUTPUT_DIR, "descriptive_statistics.csv")
    )

    print("\nNumerical Descriptive Statistics:")
    print(descriptive_stats.round(4))

# -------------------- 8. Categorical Attribute Summary -------

categorical_df = df.select_dtypes(
    include=["object", "category", "bool"]
)

if not categorical_df.empty:
    categorical_summary = pd.DataFrame({
        "Column": categorical_df.columns,
        "Unique_Categories": [
            categorical_df[col].nunique(dropna=True)
            for col in categorical_df.columns
        ]
    })

    categorical_summary.to_csv(
        os.path.join(OUTPUT_DIR, "categorical_summary.csv"),
        index=False
    )

    print("\nCategorical Attributes:")
    print(categorical_summary.to_string(index=False))

# -------------------- 9. Identify Potential Time/Region Columns

possible_time_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in
           ["year", "date", "time", "period"])
]

possible_region_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in
           ["region", "province", "city", "country", "location", "area"])
]

print("\nPotential Time Columns:", possible_time_columns)
print("Potential Regional Columns:", possible_region_columns)

# -------------------- 10. Generate Dataset Profile ------------

profile = pd.DataFrame({
    "Item": [
        "Dataset File",
        "Total Observations",
        "Total Attributes",
        "Numerical Attributes",
        "Categorical Attributes",
        "Total Missing Cells",
        "Duplicate Records"
    ],
    "Value": [
        os.path.basename(DATASET_PATH),
        len(df),
        len(df.columns),
        len(numeric_df.columns),
        len(categorical_df.columns),
        int(df.isnull().sum().sum()),
        duplicate_count
    ]
})

profile.to_csv(
    os.path.join(OUTPUT_DIR, "dataset_profile.csv"),
    index=False
)

# Save the original column names and order
pd.DataFrame({
    "Column_Order": range(1, len(df.columns) + 1),
    "Column_Name": df.columns
}).to_csv(
    os.path.join(OUTPUT_DIR, "dataset_column_list.csv"),
    index=False
)

print("\nDataset Profile:")
print(profile.to_string(index=False))

print("\n" + "=" * 65)
print("STEP 1 COMPLETED SUCCESSFULLY")
print(f"Inspection outputs saved in: {OUTPUT_DIR}")
print("=" * 65)

STEP 1: DATASET ACQUISITION AND INITIAL INSPECTION

Dataset loaded successfully.
Dataset path: G:\5287\digital_economy_gtfp_china.csv
Number of observations: 14000
Number of attributes: 38

First 10 Records:
    region  year  internet_penetration_rate  internet_users  \
0  Beijing  1985                   4.989712   863854.301573   
1  Beijing  1986                   2.481257   429572.882718   
2  Beijing  1987                   0.000000        0.000000   
3  Beijing  1988                   0.000000        0.000000   
4  Beijing  1989                   0.225669    39069.469604   
5  Beijing  1990                   0.000000        0.000000   
6  Beijing  1991                   1.071111   185438.317890   
7  Beijing  1992                   2.715961   470206.492380   
8  Beijing  1993                   0.000000        0.000000   
9  Beijing  1994                   2.311465   400177.194365   

   mobile_subscriptions  ecommerce_volume  digital_financial_inclusion_index  \
0              0.0

# DATA PREPROCESSING

In [3]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------

DATASET_PATH = r"G:\5287\digital_economy_gtfp_china.csv"

OUTPUT_DIR = r"G:\5287\Step2_Data_Preprocessing"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD DATASET
# ------------------------------------------------------------

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PATH}"
    )

df = pd.read_csv(DATASET_PATH, low_memory=False)

print("=" * 70)
print("STEP 2: DATA PREPROCESSING")
print("=" * 70)

print("\nOriginal dataset shape:", df.shape)

# Preserve original column order and names
original_columns = df.columns.tolist()

# ------------------------------------------------------------
# 3. IDENTIFY IMPORTANT COLUMNS
# ------------------------------------------------------------
# Update these names if your dataset uses different names.
# The script attempts to identify common year, region, and
# GTFP column names automatically.

def find_column(columns, candidates):
    lookup = {str(col).strip().lower(): col for col in columns}

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


year_col = find_column(
    df.columns,
    ["year", "年份", "time", "period"]
)

region_col = find_column(
    df.columns,
    [
        "region", "province", "province_name",
        "city", "地区", "省份", "区域"
    ]
)

target_col = find_column(
    df.columns,
    [
        "gtfp", "green_tfp", "green_total_factor_productivity",
        "green total factor productivity",
        "manufacturing_gtfp", "GTFP"
    ]
)

print("\nDetected year column:", year_col)
print("Detected region column:", region_col)
print("Detected GTFP target column:", target_col)

if target_col is None:
    print(
        "\nWARNING: GTFP target was not identified automatically. "
        "Set target_col manually before using this dataset "
        "for supervised prediction."
    )

# ------------------------------------------------------------
# 4. REMOVE DUPLICATE RECORDS
# ------------------------------------------------------------

initial_rows = len(df)

df = df.drop_duplicates().copy()

duplicates_removed = initial_rows - len(df)

print("\nDuplicate records removed:", duplicates_removed)
print("Shape after duplicate removal:", df.shape)

# ------------------------------------------------------------
# 5. CONVERT NUMERIC COLUMNS WHERE APPROPRIATE
# ------------------------------------------------------------
# Preserve year and region identifiers.
# Other columns are converted only when at least 90% of
# their non-missing values can be interpreted as numbers.

protected_cols = {
    col for col in [year_col, region_col]
    if col is not None
}

for col in df.columns:
    if col in protected_cols:
        continue

    if df[col].dtype == "object":
        converted = pd.to_numeric(df[col], errors="coerce")

        original_nonmissing = df[col].notna().sum()

        if original_nonmissing > 0:
            conversion_rate = (
                converted.notna().sum() / original_nonmissing
            )

            if conversion_rate >= 0.90:
                df[col] = converted

# ------------------------------------------------------------
# 6. HANDLE MISSING VALUES
# ------------------------------------------------------------

missing_before = df.isna().sum()

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing_Before": missing_before.values,
    "Missing_Percentage": (
        df.isna().mean().values * 100
    ).round(2)
})

numeric_cols = df.select_dtypes(
    include=np.number
).columns.tolist()

# Do not impute the prediction target. Missing target records
# cannot provide valid supervised-learning labels.
if target_col is not None and target_col in df.columns:
    if df[target_col].isna().any():
        target_missing = int(df[target_col].isna().sum())

        df = df.dropna(subset=[target_col]).copy()

        print(
            "\nRecords removed due to missing GTFP target:",
            target_missing
        )

# Recalculate numeric columns after any row removal.
numeric_cols = df.select_dtypes(
    include=np.number
).columns.tolist()

# Interpolate numeric features within each region, in year order,
# when both identifiers are available.
if (
    year_col is not None
    and region_col is not None
    and year_col in df.columns
    and region_col in df.columns
):
    df[year_col] = pd.to_numeric(
        df[year_col], errors="coerce"
    )

    df = df.sort_values(
        [region_col, year_col],
        na_position="last"
    ).copy()

    for col in numeric_cols:
        if col in protected_cols or col == target_col:
            continue

        df[col] = df.groupby(
            region_col, dropna=False
        )[col].transform(
            lambda s: s.interpolate(
                method="linear",
                limit_area="inside"
            )
        )

# Remaining numeric feature gaps: median imputation.
# This is a provisional step; for rigorous predictive evaluation,
# fit imputation values on training data only.
for col in numeric_cols:
    if col in protected_cols or col == target_col:
        continue

    if df[col].isna().any():
        median_value = df[col].median()

        if pd.notna(median_value):
            df[col] = df[col].fillna(median_value)

# Categorical missing values: explicit category.
for col in df.select_dtypes(
    include=["object", "category", "string"]
).columns:
    if col not in protected_cols and df[col].isna().any():
        df[col] = df[col].fillna("Unknown")

missing_after = df.isna().sum()

missing_report["Missing_After"] = [
    int(missing_after.get(col, 0))
    for col in missing_report["Column"]
]

missing_report.to_csv(
    os.path.join(OUTPUT_DIR, "missing_value_treatment.csv"),
    index=False
)

print("\nTotal missing cells before treatment:",
      int(missing_before.sum()))

print("Total missing cells after treatment:",
      int(df.isna().sum().sum()))

# ------------------------------------------------------------
# 7. IDENTIFY NUMERIC FEATURES FOR IQR ANALYSIS
# ------------------------------------------------------------

numeric_cols = df.select_dtypes(
    include=np.number
).columns.tolist()

# Exclude identifiers and target from outlier removal.
# GTFP values are preserved rather than automatically removed.
excluded_from_iqr = {
    col for col in [year_col, target_col]
    if col is not None
}

feature_cols = [
    col for col in numeric_cols
    if col not in excluded_from_iqr
]

# ------------------------------------------------------------
# 8. DETECT OUTLIERS USING IQR
# ------------------------------------------------------------
# Instead of deleting entire rows, cap feature values at
# their IQR boundaries. This preserves regional-year records.
# IQR boundaries are learned from the training partition below
# in a rigorous forecasting workflow; here, this section
# creates an initial preprocessing dataset for inspection.

iqr_report = []

for col in feature_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (df[col] < lower_bound)
        | (df[col] > upper_bound)
    )

    outlier_count = int(outlier_mask.sum())

    df[col] = df[col].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    iqr_report.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count_Capped": outlier_count
    })

iqr_df = pd.DataFrame(iqr_report)

iqr_df.to_csv(
    os.path.join(OUTPUT_DIR, "iqr_outlier_report.csv"),
    index=False
)

print("\nIQR-based outlier treatment completed.")

# ------------------------------------------------------------
# 9. DEFINE THE DATA SPLIT
# ------------------------------------------------------------
# Chronological splitting is preferred for forecasting.
# With a year column, the latest years are reserved for testing.
# Without a year column, a reproducible row-based split is used.

if year_col is not None and df[year_col].notna().any():

    df = df.sort_values(
        [year_col] +
        ([region_col] if region_col is not None else [])
    ).reset_index(drop=True)

    unique_years = sorted(
        df[year_col].dropna().unique()
    )

    if len(unique_years) >= 3:
        n_years = len(unique_years)

        train_end = max(1, int(n_years * 0.70))
        val_end = max(
            train_end + 1,
            int(n_years * 0.85)
        )

        val_end = min(val_end, n_years - 1)

        train_years = unique_years[:train_end]
        val_years = unique_years[train_end:val_end]
        test_years = unique_years[val_end:]

        train_df = df[df[year_col].isin(train_years)].copy()
        val_df = df[df[year_col].isin(val_years)].copy()
        test_df = df[df[year_col].isin(test_years)].copy()

        split_method = "Chronological year-based split"

    else:
        raise ValueError(
            "Too few distinct years for a chronological "
            "train/validation/test split. Review the dataset."
        )

else:
    # Fallback only: use this for non-temporal data.
    # Do not treat this as a valid forecasting split.
    from sklearn.model_selection import train_test_split

    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        random_state=42,
        shuffle=True
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    split_method = "Random row-based split (no year column detected)"

print("\nSplit method:", split_method)
print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Testing records:", len(test_df))

# ------------------------------------------------------------
# 10. MIN-MAX NORMALIZATION
# ------------------------------------------------------------
# Fit the scaler ONLY on the training data to reduce leakage.
# Apply the same transformation to validation and test sets.

numeric_cols = df.select_dtypes(
    include=np.number
).columns.tolist()

scale_cols = [
    col for col in numeric_cols
    if col not in protected_cols
]

# Keep GTFP target in original units for interpretable
# MAE/RMSE reporting. Scale only predictor features.
if target_col is not None and target_col in scale_cols:
    scale_cols.remove(target_col)

if scale_cols:
    scaler = MinMaxScaler()

    train_df.loc[:, scale_cols] = scaler.fit_transform(
        train_df[scale_cols]
    )

    val_df.loc[:, scale_cols] = scaler.transform(
        val_df[scale_cols]
    )

    test_df.loc[:, scale_cols] = scaler.transform(
        test_df[scale_cols]
    )

    # Save scaling parameters for reproducibility.
    scaler_parameters = pd.DataFrame({
        "Feature": scale_cols,
        "Min": scaler.data_min_,
        "Max": scaler.data_max_,
        "Scale": scaler.scale_,
        "Offset": scaler.min_
    })

    scaler_parameters.to_csv(
        os.path.join(OUTPUT_DIR, "minmax_scaler_parameters.csv"),
        index=False
    )

# ------------------------------------------------------------
# 11. SAVE PROCESSED DATASETS
# ------------------------------------------------------------

train_df.to_csv(
    os.path.join(OUTPUT_DIR, "train_preprocessed.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(OUTPUT_DIR, "validation_preprocessed.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(OUTPUT_DIR, "test_preprocessed.csv"),
    index=False
)

# Combined dataset, retaining split labels.
train_export = train_df.copy()
val_export = val_df.copy()
test_export = test_df.copy()

train_export["Dataset_Split"] = "Train"
val_export["Dataset_Split"] = "Validation"
test_export["Dataset_Split"] = "Test"

combined_df = pd.concat(
    [train_export, val_export, test_export],
    ignore_index=True
)

combined_df.to_csv(
    os.path.join(OUTPUT_DIR, "all_preprocessed_with_split.csv"),
    index=False
)

# ------------------------------------------------------------
# 12. SAVE PREPROCESSING SUMMARY
# ------------------------------------------------------------

summary = {
    "input_file": DATASET_PATH,
    "original_rows": int(initial_rows),
    "rows_after_duplicate_removal": int(initial_rows - duplicates_removed),
    "duplicates_removed": int(duplicates_removed),
    "final_rows": int(len(df)),
    "number_of_columns": int(len(df.columns)),
    "training_records": int(len(train_df)),
    "validation_records": int(len(val_df)),
    "testing_records": int(len(test_df)),
    "split_method": split_method,
    "year_column": year_col,
    "region_column": region_col,
    "target_column": target_col,
    "missing_cells_after_treatment": int(df.isna().sum().sum()),
    "normalized_predictor_columns": scale_cols
}

with open(
    os.path.join(OUTPUT_DIR, "preprocessing_summary.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

# ------------------------------------------------------------
# 13. VISUALIZE SPLIT DISTRIBUTION
# ------------------------------------------------------------

split_names = ["Training", "Validation", "Testing"]

split_counts = [
    len(train_df),
    len(val_df),
    len(test_df)
]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    split_names,
    split_counts,
    edgecolor="black"
)

plt.xlabel("Dataset Partition")
plt.ylabel("Number of Records")
plt.title("Training, Validation, and Testing Data Distribution")

for bar, count in zip(bars, split_counts):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(count),
        ha="center",
        va="bottom"
    )

plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "dataset_split_distribution.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ------------------------------------------------------------
# 14. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 COMPLETED")
print("=" * 70)

print("\nFinal dataset shape:", df.shape)

print("\nPartition sizes:")
print(f"Training:   {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Testing:    {len(test_df)}")

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", filename)

print("\nPreprocessing finished successfully.")

STEP 2: DATA PREPROCESSING

Original dataset shape: (14000, 38)

Detected year column: year
Detected region column: region
Detected GTFP target column: GTFP

Duplicate records removed: 0
Shape after duplicate removal: (14000, 38)

Total missing cells before treatment: 0
Total missing cells after treatment: 0

IQR-based outlier treatment completed.

Split method: Chronological year-based split
Training records: 9800
Validation records: 2100
Testing records: 2100

STEP 2 COMPLETED

Final dataset shape: (14000, 38)

Partition sizes:
Training:   9800
Validation: 2100
Testing:    2100

Output directory:
G:\5287\Step2_Data_Preprocessing

Generated files:
 - all_preprocessed_with_split.csv
 - dataset_split_distribution.png
 - iqr_outlier_report.csv
 - minmax_scaler_parameters.csv
 - missing_value_treatment.csv
 - preprocessing_summary.json
 - test_preprocessed.csv
 - train_preprocessed.csv
 - validation_preprocessed.csv

Preprocessing finished successfully.


# FEATURE EXTRACTION

In [5]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ============================================================
# STEP 3: KPCA FEATURE EXTRACTION
# Digital Economy-Driven Prediction of Manufacturing Green TFP
# ============================================================

INPUT_DIR = r"G:\5287\Step2_Data_Preprocessing"
OUTPUT_DIR = r"G:\5287\Step3_KPCA_Feature_Extraction"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(INPUT_DIR, "train_preprocessed.csv")
VAL_PATH = os.path.join(INPUT_DIR, "validation_preprocessed.csv")
TEST_PATH = os.path.join(INPUT_DIR, "test_preprocessed.csv")

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file not found: {path}")

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("=" * 70)
print("STEP 3: KPCA FEATURE EXTRACTION")
print("=" * 70)

print("Training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)
print("Testing dataset:", test_df.shape)

# ------------------------------------------------------------
# 2. IDENTIFY TARGET, YEAR AND REGION COLUMNS
# ------------------------------------------------------------

def find_column(columns, candidates):
    lookup = {str(col).strip().lower(): col for col in columns}

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


all_columns = train_df.columns.tolist()

year_col = find_column(
    all_columns,
    ["year", "年份", "time", "period"]
)

region_col = find_column(
    all_columns,
    ["region", "province", "province_name", "city",
     "地区", "省份", "区域"]
)

target_col = find_column(
    all_columns,
    ["gtfp", "green_tfp",
     "green_total_factor_productivity",
     "green total factor productivity",
     "manufacturing_gtfp"]
)

# If automatic target detection fails, set the exact column name:
# target_col = "GTFP"

print("\nDetected year column:", year_col)
print("Detected region column:", region_col)
print("Detected target column:", target_col)

# ------------------------------------------------------------
# 3. SELECT NUMERICAL PREDICTORS
# ------------------------------------------------------------

numeric_columns = train_df.select_dtypes(
    include=np.number
).columns.tolist()

excluded_columns = {
    col for col in [year_col, target_col]
    if col is not None
}

if region_col is not None:
    excluded_columns.add(region_col)

feature_cols = [
    col for col in numeric_columns
    if col not in excluded_columns
]

if not feature_cols:
    raise ValueError("No numerical predictor columns were identified.")

print("\nNumber of predictor variables:", len(feature_cols))
print("Selected predictors:", feature_cols)

for name, data in [
    ("Validation", val_df),
    ("Testing", test_df)
]:
    missing_features = [
        col for col in feature_cols
        if col not in data.columns
    ]

    if missing_features:
        raise ValueError(
            f"{name} dataset is missing predictors: {missing_features}"
        )

# ------------------------------------------------------------
# 4. EXTRACT FEATURE MATRICES
# ------------------------------------------------------------

X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

for name, X in [
    ("Training", X_train),
    ("Validation", X_val),
    ("Testing", X_test)
]:
    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError(
            f"{name} predictors contain missing or infinite values. "
            "Resolve them during preprocessing."
        )

# ------------------------------------------------------------
# 5. STANDARDIZE FEATURES
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

scaling_report = pd.DataFrame({
    "Feature": feature_cols,
    "Training_Mean": scaler.mean_,
    "Training_Scale": scaler.scale_
})

scaling_report.to_csv(
    os.path.join(OUTPUT_DIR, "kpca_input_scaling_parameters.csv"),
    index=False
)

# ------------------------------------------------------------
# 6. CONFIGURE PCA
# ------------------------------------------------------------

REQUESTED_COMPONENTS = 10

n_components = min(
    REQUESTED_COMPONENTS,
    X_train_scaled.shape[1],
    X_train_scaled.shape[0]
)

if n_components < 1:
    raise ValueError("Insufficient data for KPCA.")

pca = PCA(
    n_components=n_components,
    svd_solver="auto",
    random_state=42
)

# ------------------------------------------------------------
# 7. FIT PCA ON TRAINING DATA ONLY
# ------------------------------------------------------------

print("\nFitting KPCA on training predictors...")

X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("KPCA transformation completed.")
print("Original predictor count:", len(feature_cols))
print("Extracted component count:", X_train_pca.shape[1])

print("\nTransformed dataset shapes:")
print("Training:", X_train_pca.shape)
print("Validation:", X_val_pca.shape)
print("Testing:", X_test_pca.shape)

# ------------------------------------------------------------
# 8. SAVE PCA-TRANSFORMED DATASETS
# ------------------------------------------------------------

component_names = [
    f"KPCA_Component_{i + 1}"
    for i in range(X_train_pca.shape[1])
]


def create_transformed_dataset(
    original_df,
    transformed_features,
    component_names
):
    result = pd.DataFrame(
        transformed_features,
        columns=component_names,
        index=original_df.index
    )

    # Preserve year, region and target information.
    for col in [year_col, region_col, target_col]:
        if col is not None and col in original_df.columns:
            result[col] = original_df[col].to_numpy()

    metadata_cols = [
        col for col in [year_col, region_col, target_col]
        if col is not None and col in result.columns
    ]

    remaining_cols = [
        col for col in result.columns
        if col not in metadata_cols
    ]

    return result[metadata_cols + remaining_cols]


train_pca_df = create_transformed_dataset(
    train_df, X_train_pca, component_names
)

val_pca_df = create_transformed_dataset(
    val_df, X_val_pca, component_names
)

test_pca_df = create_transformed_dataset(
    test_df, X_test_pca, component_names
)

train_pca_df.to_csv(
    os.path.join(OUTPUT_DIR, "train_kpca.csv"),
    index=False
)

val_pca_df.to_csv(
    os.path.join(OUTPUT_DIR, "validation_kpca.csv"),
    index=False
)

test_pca_df.to_csv(
    os.path.join(OUTPUT_DIR, "test_kpca.csv"),
    index=False
)

# ------------------------------------------------------------
# 9. EXPLAINED VARIANCE ANALYSIS
# ------------------------------------------------------------

explained_variance = pca.explained_variance_
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_explained_variance = np.cumsum(
    explained_variance_ratio
)

variance_df = pd.DataFrame({
    "Component": component_names,
    "Eigenvalue": explained_variance,
    "Explained_Variance_Ratio_Percent":
        explained_variance_ratio * 100,
    "Cumulative_Explained_Variance_Percent":
        cumulative_explained_variance * 100
})

variance_df.to_csv(
    os.path.join(OUTPUT_DIR, "kpca_explained_variance_analysis.csv"),
    index=False
)

print("\nExplained variance by component:")
print(variance_df.to_string(index=False))

print(
    "\nCumulative explained variance:",
    f"{cumulative_explained_variance[-1] * 100:.2f}%"
)

# ------------------------------------------------------------
# 10. CUMULATIVE EXPLAINED VARIANCE PLOT
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    np.arange(1, n_components + 1),
    cumulative_explained_variance * 100,
    marker="o",
    linestyle="--",
    linewidth=2
)

plt.xlabel("Principal Component")
plt.ylabel("Cumulative Explained Variance (%)")
plt.title("KPCA Cumulative Explained Variance")
plt.xticks(np.arange(1, n_components + 1))
plt.ylim(0, 105)
plt.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "Kpca_cumulative_explained_variance.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ------------------------------------------------------------
# 11. COMPONENT-WISE EXPLAINED VARIANCE PLOT
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.bar(
    np.arange(1, n_components + 1),
    explained_variance_ratio * 100,
    edgecolor="black"
)

plt.xlabel("Principal Component")
plt.ylabel("Explained Variance (%)")
plt.title("Explained Variance of KPCA Components")
plt.xticks(np.arange(1, n_components + 1))
plt.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "kpca_explained_variance_by_component.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ------------------------------------------------------------
# 12. TWO-DIMENSIONAL PCA PROJECTION
# ------------------------------------------------------------

if X_train_pca.shape[1] >= 2:

    plt.figure(figsize=(10, 7))

    plt.scatter(
        X_train_pca[:, 0],
        X_train_pca[:, 1],
        s=28,
        alpha=0.75,
        edgecolors="none"
    )

    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title("Two-Dimensional Projection Using KPCA")
    plt.grid(True, linestyle=":", alpha=0.4)
    plt.tight_layout()

    plt.savefig(
        os.path.join(OUTPUT_DIR, "kpca_2d_projection.png"),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

# ------------------------------------------------------------
# 13. PCA SUMMARY
# ------------------------------------------------------------

summary = pd.DataFrame({
    "Measure": [
        "Original predictor count",
        "KPCA component count",
        "Training observations",
        "Validation observations",
        "Testing observations",
        "KPCA method",
        "Total explained variance (%)",
        "Target column",
        "Year column",
        "Region column"
    ],
    "Value": [
        len(feature_cols),
        X_train_pca.shape[1],
        len(train_pca_df),
        len(val_pca_df),
        len(test_pca_df),
        "Linear PCA",
        round(cumulative_explained_variance[-1] * 100, 4),
        target_col if target_col else "Not detected",
        year_col if year_col else "Not detected",
        region_col if region_col else "Not detected"
    ]
})

summary.to_csv(
    os.path.join(OUTPUT_DIR, "kpca_summary.csv"),
    index=False
)

print("\nKPCA summary:")
print(summary.to_string(index=False))

# ------------------------------------------------------------
# 14. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3 KPCA COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", filename)

print("\nKPCA-transformed datasets are ready for the next workflow stage.")

STEP 3: KPCA FEATURE EXTRACTION
Training dataset: (9800, 38)
Validation dataset: (2100, 38)
Testing dataset: (2100, 38)

Detected year column: year
Detected region column: region
Detected target column: GTFP

Number of predictor variables: 34
Selected predictors: ['internet_penetration_rate', 'internet_users', 'mobile_subscriptions', 'ecommerce_volume', 'digital_financial_inclusion_index', 'ict_investment', 'tech_firms_count', 'broadband_access_rate', 'digital_rd_expenditure', 'digital_patents', 'manufacturing_output', 'industrial_value_added', 'manufacturing_firms_count', 'manufacturing_employment', 'manufacturing_capital_stock', 'labor_productivity', 'technology_adoption_rate', 'co2_emissions', 'so2_emissions', 'nox_emissions', 'industrial_wastewater', 'solid_waste', 'energy_consumption', 'renewable_energy_share', 'green_patents', 'environmental_investment', 'gdp_per_capita', 'population', 'urbanization_rate', 'fdi', 'trade_openness', 'education_level', 'government_expenditure', 'GTF

In [6]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import euclidean_distances

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

INPUT_DIR = r"G:\5287\Step3_KPCA_Feature_Extraction"
OUTPUT_DIR = r"G:\5287\Step4_Spatial_Graph_Construction"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(INPUT_DIR, "train_kpca.csv")
VAL_PATH = os.path.join(INPUT_DIR, "validation_kpca.csv")
TEST_PATH = os.path.join(INPUT_DIR, "test_kpca.csv")

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required Step 3 file not found: {path}"
        )

# ------------------------------------------------------------
# 2. LOAD KPCA DATASETS
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("=" * 75)
print("STEP 4: SPATIAL GRAPH CONSTRUCTION")
print("=" * 75)

print("Training data:", train_df.shape)
print("Validation data:", val_df.shape)
print("Testing data:", test_df.shape)

# ------------------------------------------------------------
# 3. IDENTIFY REGION, YEAR, TARGET, AND KPCA FEATURES
# ------------------------------------------------------------

def find_column(columns, candidates):
    lookup = {
        str(col).strip().lower(): col
        for col in columns
    }

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


all_columns = train_df.columns.tolist()

# Automatically detect common column names.
REGION_COL = find_column(
    all_columns,
    [
        "region", "province", "province_name",
        "city", "地区", "省份", "区域"
    ]
)

YEAR_COL = find_column(
    all_columns,
    ["year", "年份", "time", "period"]
)

TARGET_COL = find_column(
    all_columns,
    [
        "gtfp",
        "green_tfp",
        "green_total_factor_productivity",
        "green total factor productivity",
        "manufacturing_gtfp"
    ]
)

# If detection fails, set these manually to the exact CSV headers.
# Example:
# REGION_COL = "Province"
# YEAR_COL = "Year"
# TARGET_COL = "GTFP"

if REGION_COL is None:
    raise ValueError(
        "Region column was not detected. Set REGION_COL "
        "manually to the exact region/province column name."
    )

if YEAR_COL is None:
    raise ValueError(
        "Year column was not detected. Set YEAR_COL "
        "manually to the exact time column name."
    )

if REGION_COL not in train_df.columns:
    raise ValueError(f"Region column not found: {REGION_COL}")

if YEAR_COL not in train_df.columns:
    raise ValueError(f"Year column not found: {YEAR_COL}")

# KPCA features created in Step 3.
FEATURE_COLS = [
    col for col in train_df.columns
    if str(col).startswith("KPCA_Component_")
]

if not FEATURE_COLS:
    raise ValueError(
        "No KPCA components were found. Check the Step 3 output."
    )

for name, frame in [
    ("Validation", val_df),
    ("Testing", test_df)
]:
    missing = [
        col for col in FEATURE_COLS
        if col not in frame.columns
    ]

    if missing:
        raise ValueError(
            f"{name} data are missing KPCA features: {missing}"
        )

print("\nRegion column:", REGION_COL)
print("Year column:", YEAR_COL)
print("Target column:", TARGET_COL)
print("Number of KPCA features:", len(FEATURE_COLS))

# ------------------------------------------------------------
# 4. STANDARDIZE REGION AND YEAR IDENTIFIERS
# ------------------------------------------------------------

for frame in [train_df, val_df, test_df]:

    frame[REGION_COL] = (
        frame[REGION_COL]
        .astype("string")
        .str.strip()
    )

    frame[YEAR_COL] = pd.to_numeric(
        frame[YEAR_COL],
        errors="coerce"
    )

    frame[FEATURE_COLS] = frame[FEATURE_COLS].apply(
        pd.to_numeric,
        errors="coerce"
    )

    frame.dropna(
        subset=[REGION_COL, YEAR_COL],
        inplace=True
    )

    if frame[FEATURE_COLS].isna().any().any():
        raise ValueError(
            "KPCA features contain missing or invalid values. "
            "Inspect the Step 3 output before proceeding."
        )

# Use the same region list for all partitions.
regions = sorted(
    set(train_df[REGION_COL].unique())
    | set(val_df[REGION_COL].unique())
    | set(test_df[REGION_COL].unique())
)

region_to_index = {
    region: idx
    for idx, region in enumerate(regions)
}

num_regions = len(regions)
num_features = len(FEATURE_COLS)

if num_regions < 2:
    raise ValueError(
        "At least two distinct regions are required "
        "to construct an interregional graph."
    )

print("\nNumber of regional nodes:", num_regions)
print("Number of KPCA features per node:", num_features)

# ------------------------------------------------------------
# 5. CONSTRUCT REGIONAL NODE REPRESENTATIONS
# ------------------------------------------------------------
# Each node is represented by the mean of its training-period
# KPCA features.
#
# Validation/test observations are NOT used to estimate
# the adjacency matrix.

regional_features = (
    train_df.groupby(REGION_COL)[FEATURE_COLS]
    .mean()
    .reindex(regions)
)

# Regions without training-period observations cannot receive
# data-derived similarity weights from this procedure.
missing_training_regions = regional_features[
    regional_features.isna().any(axis=1)
].index.tolist()

if missing_training_regions:
    raise ValueError(
        "These regions have no usable training-period features: "
        f"{missing_training_regions}. Review the dataset split."
    )

regional_matrix = regional_features.to_numpy(dtype=float)

# Save region index mapping.
region_mapping = pd.DataFrame({
    "Node_Index": np.arange(num_regions),
    "Region": regions
})

region_mapping.to_csv(
    os.path.join(OUTPUT_DIR, "regional_node_mapping.csv"),
    index=False
)

# Save regional representations.
regional_features.to_csv(
    os.path.join(OUTPUT_DIR, "regional_node_features.csv"),
    index_label="Region"
)

# ------------------------------------------------------------
# 6. BUILD THE WEIGHTED ADJACENCY MATRIX
# ------------------------------------------------------------
# Use an RBF-style feature-similarity kernel:
#
# A_ij = exp(-d_ij^2 / (2*sigma^2))
#
# where d_ij is Euclidean distance between regional mean
# KPCA representations.
#
# The bandwidth is estimated from positive pairwise distances.
# This is a feature-similarity graph, not a geographic graph.

distances = euclidean_distances(regional_matrix)

upper_triangle = distances[
    np.triu_indices(num_regions, k=1)
]

positive_distances = upper_triangle[
    upper_triangle > 0
]

if len(positive_distances) == 0:
    raise ValueError(
        "All regional representations are identical. "
        "A meaningful similarity graph cannot be constructed."
    )

sigma = float(np.median(positive_distances))

if sigma <= 0:
    raise ValueError(
        "Invalid similarity bandwidth."
    )

adjacency = np.exp(
    -(distances ** 2) / (2 * sigma ** 2)
)

# Remove self-connections before applying the threshold.
np.fill_diagonal(adjacency, 0.0)

# Optional sparsification:
# Keep only the strongest K regional connections per node.
# This reduces weak, noisy connections.
TOP_K_NEIGHBORS = min(5, num_regions - 1)

sparse_adjacency = np.zeros_like(adjacency)

for i in range(num_regions):

    neighbor_indices = np.argsort(
        adjacency[i]
    )[-TOP_K_NEIGHBORS:]

    sparse_adjacency[i, neighbor_indices] = (
        adjacency[i, neighbor_indices]
    )

# Make the graph undirected by symmetrizing the weights.
adjacency = np.maximum(
    sparse_adjacency,
    sparse_adjacency.T
)

# Add self-loops.
adjacency_with_self_loops = (
    adjacency + np.eye(num_regions)
)

# ------------------------------------------------------------
# 7. NORMALIZE THE ADJACENCY MATRIX
# ------------------------------------------------------------
# Symmetric normalization:
#
# A_norm = D^(-1/2) * A_tilde * D^(-1/2)
#
# A_tilde includes self-loops.

degree = adjacency_with_self_loops.sum(axis=1)

inverse_sqrt_degree = np.power(
    degree,
    -0.5,
    where=degree > 0
)

inverse_sqrt_degree[degree <= 0] = 0.0

D_inv_sqrt = np.diag(inverse_sqrt_degree)

normalized_adjacency = (
    D_inv_sqrt
    @ adjacency_with_self_loops
    @ D_inv_sqrt
)

# ------------------------------------------------------------
# 8. SAVE ADJACENCY MATRICES
# ------------------------------------------------------------

pd.DataFrame(
    adjacency,
    index=regions,
    columns=regions
).to_csv(
    os.path.join(OUTPUT_DIR, "adjacency_matrix.csv"),
    index_label="Region"
)

pd.DataFrame(
    adjacency_with_self_loops,
    index=regions,
    columns=regions
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "adjacency_matrix_with_self_loops.csv"
    ),
    index_label="Region"
)

pd.DataFrame(
    normalized_adjacency,
    index=regions,
    columns=regions
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "normalized_adjacency_matrix.csv"
    ),
    index_label="Region"
)

np.save(
    os.path.join(OUTPUT_DIR, "adjacency_matrix.npy"),
    adjacency
)

np.save(
    os.path.join(
        OUTPUT_DIR,
        "normalized_adjacency_matrix.npy"
    ),
    normalized_adjacency
)

# ------------------------------------------------------------
# 9. GENERATE EDGE LIST
# ------------------------------------------------------------

edges = []

for i in range(num_regions):
    for j in range(i + 1, num_regions):

        weight = adjacency[i, j]

        if weight > 0:
            edges.append({
                "Source_Node": i,
                "Source_Region": regions[i],
                "Target_Node": j,
                "Target_Region": regions[j],
                "Edge_Weight": float(weight)
            })

edge_df = pd.DataFrame(
    edges,
    columns=[
        "Source_Node",
        "Source_Region",
        "Target_Node",
        "Target_Region",
        "Edge_Weight"
    ]
)

edge_df.to_csv(
    os.path.join(OUTPUT_DIR, "regional_edge_list.csv"),
    index=False
)

# ------------------------------------------------------------
# 10. CREATE REGION-YEAR FEATURE TENSORS
# ------------------------------------------------------------
# Output tensor shape:
# (number of years, number of regions, number of KPCA features)
#
# A corresponding observation mask identifies missing region-year
# combinations. Missing observations are filled with zero only
# for tensor storage; the mask must be used by downstream
# processing to distinguish missing records from real values.

def build_temporal_tensor(frame, split_name):

    if frame.empty:
        raise ValueError(
            f"{split_name} dataset is empty."
        )

    # Aggregate duplicate region-year records if they exist.
    # The target is averaged only for duplicate region-year pairs.
    aggregation = {
        col: "mean"
        for col in FEATURE_COLS
    }

    if TARGET_COL is not None and TARGET_COL in frame.columns:
        aggregation[TARGET_COL] = "mean"

    grouped = (
        frame.groupby(
            [YEAR_COL, REGION_COL],
            as_index=False
        )
        .agg(aggregation)
    )

    years = np.sort(
        grouped[YEAR_COL].unique()
    )

    year_to_index = {
        year: idx
        for idx, year in enumerate(years)
    }

    tensor = np.zeros(
        (len(years), num_regions, num_features),
        dtype=np.float32
    )

    observation_mask = np.zeros(
        (len(years), num_regions),
        dtype=np.uint8
    )

    target_tensor = None

    if TARGET_COL is not None and TARGET_COL in grouped.columns:
        target_tensor = np.full(
            (len(years), num_regions),
            np.nan,
            dtype=np.float32
        )

    for _, row in grouped.iterrows():

        region = row[REGION_COL]
        year = row[YEAR_COL]

        if region not in region_to_index:
            continue

        t = year_to_index[year]
        n = region_to_index[region]

        tensor[t, n, :] = row[
            FEATURE_COLS
        ].to_numpy(dtype=np.float32)

        observation_mask[t, n] = 1

        if target_tensor is not None:
            target_tensor[t, n] = float(
                row[TARGET_COL]
            )

    # Save each temporal partition.
    save_path = os.path.join(
        OUTPUT_DIR,
        f"{split_name.lower()}_graph_tensor.npz"
    )

    save_data = {
        "features": tensor,
        "observation_mask": observation_mask,
        "years": years,
        "regions": np.asarray(regions, dtype=str),
        "feature_names": np.asarray(FEATURE_COLS, dtype=str)
    }

    if target_tensor is not None:
        save_data["target"] = target_tensor

    np.savez_compressed(
        save_path,
        **save_data
    )

    # Save a long-form table for inspection.
    tensor_records = []

    for t, year in enumerate(years):
        for n, region in enumerate(regions):

            record = {
                "Year": year,
                "Region": region,
                "Node_Index": n,
                "Observed": int(observation_mask[t, n])
            }

            for f, feature_name in enumerate(FEATURE_COLS):
                record[feature_name] = tensor[t, n, f]

            if target_tensor is not None:
                record[TARGET_COL] = target_tensor[t, n]

            tensor_records.append(record)

    pd.DataFrame(tensor_records).to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"{split_name.lower()}_region_year_features.csv"
        ),
        index=False
    )

    print(
        f"\n{split_name} tensor shape: {tensor.shape}"
    )

    print(
        f"{split_name} observed region-year cells: "
        f"{int(observation_mask.sum())}"
    )

    print(
        f"{split_name} missing region-year cells: "
        f"{int(observation_mask.size - observation_mask.sum())}"
    )

    return tensor, observation_mask, years


train_tensor, train_mask, train_years = (
    build_temporal_tensor(train_df, "Training")
)

val_tensor, val_mask, val_years = (
    build_temporal_tensor(val_df, "Validation")
)

test_tensor, test_mask, test_years = (
    build_temporal_tensor(test_df, "Testing")
)

# ------------------------------------------------------------
# 11. VISUALIZE THE ADJACENCY MATRIX
# ------------------------------------------------------------

plt.figure(figsize=(11, 9))

image = plt.imshow(
    adjacency,
    aspect="auto",
    interpolation="nearest"
)

plt.colorbar(
    image,
    label="Regional Similarity Weight"
)

plt.xticks(
    np.arange(num_regions),
    regions,
    rotation=90
)

plt.yticks(
    np.arange(num_regions),
    regions
)

plt.xlabel("Regional Nodes")
plt.ylabel("Regional Nodes")
plt.title("Weighted Regional Adjacency Matrix")

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "regional_adjacency_heatmap.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ------------------------------------------------------------
# 12. VISUALIZE THE GRAPH DEGREE DISTRIBUTION
# ------------------------------------------------------------

degree_values = (
    adjacency > 0
).sum(axis=1)

plt.figure(figsize=(10, 6))

plt.bar(
    np.arange(num_regions),
    degree_values,
    edgecolor="black"
)

plt.xlabel("Regional Node Index")
plt.ylabel("Number of Connected Neighbors")
plt.title("Regional Graph Connectivity Distribution")

plt.xticks(
    np.arange(num_regions),
    regions,
    rotation=90
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "regional_graph_degree_distribution.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ------------------------------------------------------------
# 13. GRAPH STATISTICS
# ------------------------------------------------------------

number_of_edges = int(
    np.count_nonzero(np.triu(adjacency, k=1))
)

possible_edges = num_regions * (num_regions - 1) / 2

graph_density = (
    number_of_edges / possible_edges
    if possible_edges > 0 else 0
)

graph_summary = {
    "number_of_regions": int(num_regions),
    "number_of_kpca_features": int(num_features),
    "number_of_undirected_edges": number_of_edges,
    "possible_undirected_edges": int(possible_edges),
    "graph_density": float(graph_density),
    "similarity_kernel": "RBF",
    "similarity_bandwidth": float(sigma),
    "top_k_neighbors": int(TOP_K_NEIGHBORS),
    "self_loops_added": True,
    "normalization": "Symmetric D^(-1/2) A D^(-1/2)",
    "region_column": REGION_COL,
    "year_column": YEAR_COL,
    "target_column": TARGET_COL
}

with open(
    os.path.join(OUTPUT_DIR, "graph_summary.json"),
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        graph_summary,
        file,
        indent=4,
        ensure_ascii=False
    )

print("\nGraph summary:")

for key, value in graph_summary.items():
    print(f"{key}: {value}")

# ------------------------------------------------------------
# 14. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 4 COMPLETED")
print("=" * 75)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", filename)

print("\nSpatial graph construction completed.")
print("Graph tensors are prepared for the STGCN stage.")

STEP 4: SPATIAL GRAPH CONSTRUCTION
Training data: (9800, 13)
Validation data: (2100, 13)
Testing data: (2100, 13)

Region column: region
Year column: year
Target column: GTFP
Number of KPCA features: 10

Number of regional nodes: 350
Number of KPCA features per node: 10

Training tensor shape: (28, 350, 10)
Training observed region-year cells: 9800
Training missing region-year cells: 0

Validation tensor shape: (6, 350, 10)
Validation observed region-year cells: 2100
Validation missing region-year cells: 0

Testing tensor shape: (6, 350, 10)
Testing observed region-year cells: 2100
Testing missing region-year cells: 0

Graph summary:
number_of_regions: 350
number_of_kpca_features: 10
number_of_undirected_edges: 1101
possible_undirected_edges: 61075
graph_density: 0.018027015963978713
similarity_kernel: RBF
similarity_bandwidth: 4.002739354920778
top_k_neighbors: 5
self_loops_added: True
normalization: Symmetric D^(-1/2) A D^(-1/2)
region_column: region
year_column: year
target_column: 

# SPIDER WASP OPTIMIZATION (SWO) FOR STGCN

In [7]:
# ============================================================
# STEP 5: SPIDER WASP OPTIMIZATION (SWO) FOR STGCN
#
# Manuscript:
# Digital Economy-Driven Prediction of Manufacturing Green
# Total Factor Productivity Using SWO-STGCN
#
# Inputs:
#   training_graph_tensor.npz
#   validation_graph_tensor.npz
#   normalized_adjacency_matrix.npy
#
# Outputs:
#   Best hyperparameters
#   Optimization history
#   Fitness convergence plot
#   Best validation predictions
#
# IMPORTANT:
# The test dataset is NOT loaded or used in this step.
# ============================================================

import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

INPUT_DIR = r"G:\5287\Step4_Spatial_Graph_Construction"
OUTPUT_DIR = r"G:\5287\Step5_SWO_Optimization"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(
    INPUT_DIR, "training_graph_tensor.npz"
)

VAL_PATH = os.path.join(
    INPUT_DIR, "validation_graph_tensor.npz"
)

ADJ_PATH = os.path.join(
    INPUT_DIR, "normalized_adjacency_matrix.npy"
)

print("=" * 75)
print("STEP 5: SWO HYPERPARAMETER OPTIMIZATION")
print("=" * 75)

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. LOAD TRAINING AND VALIDATION DATA
# ------------------------------------------------------------

for path in [TRAIN_PATH, VAL_PATH, ADJ_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

train_data = np.load(TRAIN_PATH, allow_pickle=False)
val_data = np.load(VAL_PATH, allow_pickle=False)

X_train = train_data["features"].astype(np.float32)
Y_train = train_data["target"].astype(np.float32)
M_train = train_data["observation_mask"].astype(bool)

X_val = val_data["features"].astype(np.float32)
Y_val = val_data["target"].astype(np.float32)
M_val = val_data["observation_mask"].astype(bool)

A = np.load(ADJ_PATH).astype(np.float32)

regions = train_data["regions"].astype(str)
years_train = train_data["years"]
years_val = val_data["years"]

print("\nTraining feature tensor:", X_train.shape)
print("Training target tensor:", Y_train.shape)
print("Validation feature tensor:", X_val.shape)
print("Validation target tensor:", Y_val.shape)
print("Adjacency matrix:", A.shape)

# ------------------------------------------------------------
# 3. VALIDATE INPUT SHAPES
# ------------------------------------------------------------

if X_train.ndim != 3 or X_val.ndim != 3:
    raise ValueError(
        "Feature tensors must have shape "
        "(time, nodes, features)."
    )

if Y_train.shape != X_train.shape[:2]:
    raise ValueError(
        "Training target shape does not match "
        "the time and node dimensions."
    )

if Y_val.shape != X_val.shape[:2]:
    raise ValueError(
        "Validation target shape does not match "
        "the time and node dimensions."
    )

if A.shape != (X_train.shape[1], X_train.shape[1]):
    raise ValueError(
        "Adjacency matrix dimensions do not match "
        "the number of regional nodes."
    )

if not np.isfinite(X_train).all():
    raise ValueError("Training features contain invalid values.")

if not np.isfinite(X_val).all():
    raise ValueError("Validation features contain invalid values.")

if not np.isfinite(A).all():
    raise ValueError("Adjacency matrix contains invalid values.")

if not np.isfinite(Y_train[M_train]).all():
    raise ValueError("Observed training targets contain invalid values.")

if not np.isfinite(Y_val[M_val]).all():
    raise ValueError("Observed validation targets contain invalid values.")

# ------------------------------------------------------------
# 4. PREPARE TARGET SCALING
# ------------------------------------------------------------
# Scale targets using training observations only.
# This improves numerical stability during neural network
# optimization. Predictions are converted back to original
# GTFP units before calculating validation RMSE.

train_target_values = Y_train[M_train]

if len(train_target_values) == 0:
    raise ValueError("No observed training targets were found.")

TARGET_MEAN = float(np.mean(train_target_values))
TARGET_STD = float(np.std(train_target_values))

if TARGET_STD < 1e-12:
    TARGET_STD = 1.0

Y_train_scaled = (
    Y_train - TARGET_MEAN
) / TARGET_STD

Y_val_scaled = (
    Y_val - TARGET_MEAN
) / TARGET_STD

# ------------------------------------------------------------
# 5. DEFINE TEMPORAL WINDOWS
# ------------------------------------------------------------

def create_training_sequences(
    X,
    Y,
    mask,
    window
):
    """
    Predict the next year's GTFP for each region.

    Input shape:
        (window, nodes, features)

    Target shape:
        (nodes,)
    """

    X_samples = []
    Y_samples = []
    sample_masks = []

    for t in range(window, X.shape[0]):

        x_window = X[t - window:t]
        y_next = Y[t]
        valid_next = mask[t] & np.isfinite(y_next)

        X_samples.append(x_window)
        Y_samples.append(y_next)
        sample_masks.append(valid_next)

    if not X_samples:
        raise ValueError(
            f"Window {window} is too large for the training period."
        )

    return (
        np.asarray(X_samples, dtype=np.float32),
        np.asarray(Y_samples, dtype=np.float32),
        np.asarray(sample_masks, dtype=bool)
    )


def create_validation_sequences(
    X_train,
    X_val,
    Y_val,
    M_val,
    window
):
    """
    Use the last training years as historical context
    for the first validation prediction.

    No validation targets are used as input features.
    """

    context = X_train[-window:]

    combined_features = np.concatenate(
        [context, X_val],
        axis=0
    )

    X_samples = []
    Y_samples = []
    masks = []

    for i in range(X_val.shape[0]):

        target_index = window + i

        X_samples.append(
            combined_features[
                target_index - window:target_index
            ]
        )

        Y_samples.append(Y_val[i])

        masks.append(
            M_val[i] & np.isfinite(Y_val[i])
        )

    return (
        np.asarray(X_samples, dtype=np.float32),
        np.asarray(Y_samples, dtype=np.float32),
        np.asarray(masks, dtype=bool)
    )


# ------------------------------------------------------------
# 6. DEFINE GRAPH CONVOLUTION
# ------------------------------------------------------------

class GraphConvolution(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()

        self.linear = nn.Linear(
            in_features,
            out_features
        )

    def forward(self, x, adjacency):
        """
        x:
            (batch, time, nodes, features)

        adjacency:
            (nodes, nodes)
        """

        # Aggregate regional information through graph edges.
        x = torch.einsum(
            "ij,btjf->btif",
            adjacency,
            x
        )

        return self.linear(x)


# ------------------------------------------------------------
# 7. DEFINE SPATIOTEMPORAL GRAPH NETWORK
# ------------------------------------------------------------

class STGCNModel(nn.Module):

    def __init__(
        self,
        input_features,
        hidden_dim,
        dropout
    ):
        super().__init__()

        self.graph_layer = GraphConvolution(
            input_features,
            hidden_dim
        )

        self.temporal_conv = nn.Conv2d(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            kernel_size=(3, 1),
            padding=(1, 0)
        )

        self.activation = nn.GELU()

        self.dropout = nn.Dropout(dropout)

        self.output_layer = nn.Linear(
            hidden_dim,
            1
        )

    def forward(self, x, adjacency):
        """
        Input:
            (batch, time, nodes, features)

        Output:
            (batch, nodes)
        """

        # Spatial graph convolution.
        x = self.graph_layer(
            x,
            adjacency
        )

        x = self.activation(x)

        # Conv2d expects:
        # (batch, channels, time, nodes)
        x = x.permute(0, 3, 1, 2)

        # Temporal convolution.
        x = self.temporal_conv(x)

        x = self.activation(x)

        # Return to:
        # (batch, time, nodes, hidden_dim)
        x = x.permute(0, 2, 3, 1)

        # Use the final time step for next-year prediction.
        x = x[:, -1, :, :]

        x = self.dropout(x)

        output = self.output_layer(x).squeeze(-1)

        return output


# ------------------------------------------------------------
# 8. DEFINE HYPERPARAMETER SEARCH SPACE
# ------------------------------------------------------------

SEARCH_SPACE = {
    "learning_rate": [
        0.0001,
        0.0003,
        0.0005,
        0.001
    ],
    "hidden_dim": [
        16,
        32,
        64
    ],
    "dropout": [
        0.1,
        0.2,
        0.3,
        0.4
    ],
    "window": [
        2,
        3,
        4,
        5
    ],
    "weight_decay": [
        0.000001,
        0.00001,
        0.0001
    ]
}

PARAMETER_NAMES = list(SEARCH_SPACE.keys())


def random_candidate():
    return {
        key: random.choice(values)
        for key, values in SEARCH_SPACE.items()
    }


def repair_candidate(candidate):
    """
    Ensure every parameter belongs to the defined search space.
    """

    repaired = {}

    for key, allowed_values in SEARCH_SPACE.items():

        value = candidate.get(key)

        if value not in allowed_values:
            value = min(
                allowed_values,
                key=lambda v: abs(float(v) - float(value))
                if value is not None else float("inf")
            )

        repaired[key] = value

    return repaired


# ------------------------------------------------------------
# 9. EVALUATE ONE HYPERPARAMETER CONFIGURATION
# ------------------------------------------------------------

EPOCHS_PER_CANDIDATE = 12
BATCH_SIZE = 8


def evaluate_candidate(params):

    params = repair_candidate(params)

    window = int(params["window"])

    Xtr, Ytr, Mtr = create_training_sequences(
        X_train,
        Y_train_scaled,
        M_train,
        window
    )

    Xva, Yva, Mva = create_validation_sequences(
        X_train,
        X_val,
        Y_val_scaled,
        M_val,
        window
    )

    # Convert to tensors.
    Xtr_t = torch.tensor(
        Xtr,
        dtype=torch.float32,
        device=DEVICE
    )

    Ytr_t = torch.tensor(
        Ytr,
        dtype=torch.float32,
        device=DEVICE
    )

    Mtr_t = torch.tensor(
        Mtr,
        dtype=torch.bool,
        device=DEVICE
    )

    Xva_t = torch.tensor(
        Xva,
        dtype=torch.float32,
        device=DEVICE
    )

    Yva_t = torch.tensor(
        Yva,
        dtype=torch.float32,
        device=DEVICE
    )

    Mva_t = torch.tensor(
        Mva,
        dtype=torch.bool,
        device=DEVICE
    )

    A_t = torch.tensor(
        A,
        dtype=torch.float32,
        device=DEVICE
    )

    model = STGCNModel(
        input_features=X_train.shape[-1],
        hidden_dim=int(params["hidden_dim"]),
        dropout=float(params["dropout"])
    ).to(DEVICE)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=float(params["learning_rate"]),
        weight_decay=float(params["weight_decay"])
    )

    criterion = nn.MSELoss()

    n_samples = Xtr_t.shape[0]

    for epoch in range(EPOCHS_PER_CANDIDATE):

        model.train()

        permutation = torch.randperm(
            n_samples,
            device=DEVICE
        )

        for start in range(
            0,
            n_samples,
            BATCH_SIZE
        ):

            indices = permutation[
                start:start + BATCH_SIZE
            ]

            xb = Xtr_t[indices]
            yb = Ytr_t[indices]
            mb = Mtr_t[indices]

            optimizer.zero_grad()

            predictions = model(
                xb,
                A_t
            )

            if not mb.any():
                continue

            loss = criterion(
                predictions[mb],
                yb[mb]
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

    # Evaluate on validation data.
    model.eval()

    with torch.no_grad():

        predictions = model(
            Xva_t,
            A_t
        )

        valid_mask = (
            Mva_t
            & torch.isfinite(Yva_t)
        )

        if not valid_mask.any():
            raise ValueError(
                "No valid validation targets were found."
            )

        pred_original = (
            predictions[valid_mask].cpu().numpy()
            * TARGET_STD
            + TARGET_MEAN
        )

        true_original = (
            Yva_t[valid_mask].cpu().numpy()
            * TARGET_STD
            + TARGET_MEAN
        )

        rmse = float(
            np.sqrt(
                mean_squared_error(
                    true_original,
                    pred_original
                )
            )
        )

    # Free resources between candidate evaluations.
    del model, optimizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return rmse


# ------------------------------------------------------------
# 10. SWO-INSPIRED POPULATION SEARCH
# ------------------------------------------------------------
# This is an explicit discrete, SWO-inspired search procedure.
# It uses population initialization, candidate perturbation,
# exploration, exploitation, and greedy replacement.
#
# It should not be described as a verified reproduction of a
# specific published SWO variant unless its update equations
# match the exact algorithm cited in the manuscript.

POPULATION_SIZE = 8
MAX_ITERATIONS = 10

population = [
    random_candidate()
    for _ in range(POPULATION_SIZE)
]

fitness = []

print("\nInitializing optimization population...")

for i, candidate in enumerate(population):

    score = evaluate_candidate(candidate)

    fitness.append(score)

    print(
        f"Initial candidate {i + 1}/{POPULATION_SIZE} | "
        f"Validation RMSE = {score:.6f} | "
        f"Parameters = {candidate}"
    )

fitness = np.asarray(fitness, dtype=float)

best_index = int(np.argmin(fitness))

global_best = population[best_index].copy()
global_best_fitness = float(fitness[best_index])

history = []

history.append({
    "Iteration": 0,
    "Best_Validation_RMSE": global_best_fitness,
    **global_best
})

print("\nInitial best RMSE:", global_best_fitness)

# ------------------------------------------------------------
# 11. OPTIMIZATION ITERATIONS
# ------------------------------------------------------------

for iteration in range(1, MAX_ITERATIONS + 1):

    print(
        f"\nOptimization iteration "
        f"{iteration}/{MAX_ITERATIONS}"
    )

    progress = iteration / MAX_ITERATIONS

    updated_population = []
    updated_fitness = []

    for i in range(POPULATION_SIZE):

        current = population[i].copy()

        # Explore the search space with a random candidate.
        random_peer = population[
            random.randrange(POPULATION_SIZE)
        ]

        # Use the current global best as the exploitation guide.
        candidate = {}

        for key, allowed_values in SEARCH_SPACE.items():

            if random.random() < (1.0 - progress):

                # Exploration: sample another candidate value.
                candidate[key] = random.choice(
                    allowed_values
                )

            else:

                # Exploitation: move toward the current best
                # or combine information from another candidate.
                if random.random() < 0.65:
                    candidate[key] = global_best[key]
                else:
                    candidate[key] = random_peer[key]

        # Occasional mutation to maintain diversity.
        if random.random() < 0.30:

            mutation_key = random.choice(
                PARAMETER_NAMES
            )

            candidate[mutation_key] = random.choice(
                SEARCH_SPACE[mutation_key]
            )

        candidate = repair_candidate(candidate)

        candidate_fitness = evaluate_candidate(
            candidate
        )

        # Greedy replacement: retain the better candidate.
        if candidate_fitness < fitness[i]:

            updated_population.append(candidate)
            updated_fitness.append(candidate_fitness)

        else:

            updated_population.append(current)
            updated_fitness.append(fitness[i])

        print(
            f"Candidate {i + 1}/{POPULATION_SIZE} | "
            f"Candidate RMSE = {candidate_fitness:.6f} | "
            f"Best RMSE = {global_best_fitness:.6f}"
        )

    population = updated_population

    fitness = np.asarray(
        updated_fitness,
        dtype=float
    )

    best_index = int(np.argmin(fitness))

    if fitness[best_index] < global_best_fitness:

        global_best = population[best_index].copy()

        global_best_fitness = float(
            fitness[best_index]
        )

    history.append({
        "Iteration": iteration,
        "Best_Validation_RMSE": global_best_fitness,
        **global_best
    })

    print("Best parameters:", global_best)
    print(
        f"Best validation RMSE: "
        f"{global_best_fitness:.6f}"
    )

# ------------------------------------------------------------
# 12. SAVE OPTIMIZATION RESULTS
# ------------------------------------------------------------

history_df = pd.DataFrame(history)

history_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "swo_optimization_history.csv"
    ),
    index=False
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "best_swo_hyperparameters.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            "best_parameters": global_best,
            "best_validation_rmse": global_best_fitness,
            "population_size": POPULATION_SIZE,
            "maximum_iterations": MAX_ITERATIONS,
            "epochs_per_candidate": EPOCHS_PER_CANDIDATE,
            "batch_size": BATCH_SIZE,
            "fitness_function": "Validation RMSE",
            "target_mean": TARGET_MEAN,
            "target_standard_deviation": TARGET_STD,
            "seed": SEED
        },
        file,
        indent=4
    )

# ------------------------------------------------------------
# 13. PLOT FITNESS CONVERGENCE
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    history_df["Iteration"],
    history_df["Best_Validation_RMSE"],
    marker="o",
    linestyle="-",
    linewidth=2
)

plt.xlabel("Optimization Iteration")
plt.ylabel("Best Validation RMSE")
plt.title("SWO-Inspired STGCN Hyperparameter Optimization")

plt.grid(
    True,
    linestyle=":",
    alpha=0.5
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "swo_fitness_convergence.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ------------------------------------------------------------
# 14. TRAIN THE BEST CONFIGURATION
# ------------------------------------------------------------
# Retrain the selected model using training data only.
# Save its parameters for subsequent final evaluation.
#
# Validation is used only to select hyperparameters.
# The test partition remains untouched.

best_window = int(global_best["window"])

Xtr, Ytr, Mtr = create_training_sequences(
    X_train,
    Y_train_scaled,
    M_train,
    best_window
)

Xtr_t = torch.tensor(
    Xtr,
    dtype=torch.float32,
    device=DEVICE
)

Ytr_t = torch.tensor(
    Ytr,
    dtype=torch.float32,
    device=DEVICE
)

Mtr_t = torch.tensor(
    Mtr,
    dtype=torch.bool,
    device=DEVICE
)

A_t = torch.tensor(
    A,
    dtype=torch.float32,
    device=DEVICE
)

best_model = STGCNModel(
    input_features=X_train.shape[-1],
    hidden_dim=int(global_best["hidden_dim"]),
    dropout=float(global_best["dropout"])
).to(DEVICE)

best_optimizer = optim.AdamW(
    best_model.parameters(),
    lr=float(global_best["learning_rate"]),
    weight_decay=float(global_best["weight_decay"])
)

criterion = nn.MSELoss()

FINAL_TRAINING_EPOCHS = 50

final_training_history = []

for epoch in range(FINAL_TRAINING_EPOCHS):

    best_model.train()

    permutation = torch.randperm(
        Xtr_t.shape[0],
        device=DEVICE
    )

    epoch_losses = []

    for start in range(
        0,
        Xtr_t.shape[0],
        BATCH_SIZE
    ):

        indices = permutation[
            start:start + BATCH_SIZE
        ]

        xb = Xtr_t[indices]
        yb = Ytr_t[indices]
        mb = Mtr_t[indices]

        best_optimizer.zero_grad()

        predictions = best_model(
            xb,
            A_t
        )

        if not mb.any():
            continue

        loss = criterion(
            predictions[mb],
            yb[mb]
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            best_model.parameters(),
            max_norm=1.0
        )

        best_optimizer.step()

        epoch_losses.append(
            float(loss.detach().cpu())
        )

    mean_loss = (
        float(np.mean(epoch_losses))
        if epoch_losses else np.nan
    )

    final_training_history.append({
        "Epoch": epoch + 1,
        "Training_MSE": mean_loss
    })

    if (epoch + 1) % 10 == 0:

        print(
            f"Final training epoch {epoch + 1}/"
            f"{FINAL_TRAINING_EPOCHS} | "
            f"MSE = {mean_loss:.6f}"
        )

# Save model parameters.
torch.save(
    best_model.state_dict(),
    os.path.join(
        OUTPUT_DIR,
        "best_swo_stgcn_model.pth"
    )
)

pd.DataFrame(
    final_training_history
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "best_model_training_history.csv"
    ),
    index=False
)

# Save configuration needed to reload the model.
model_config = {
    "input_features": int(X_train.shape[-1]),
    "hidden_dim": int(global_best["hidden_dim"]),
    "dropout": float(global_best["dropout"]),
    "window": best_window,
    "target_mean": TARGET_MEAN,
    "target_std": TARGET_STD,
    "region_count": int(X_train.shape[1]),
    "regions": regions.tolist(),
    "feature_names": train_data["feature_names"].astype(str).tolist(),
    "seed": SEED
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "best_model_config.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_config,
        file,
        indent=4
    )

# ------------------------------------------------------------
# 15. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 5 COMPLETED")
print("=" * 75)

print("\nBest hyperparameters:")

for key, value in global_best.items():
    print(f"{key}: {value}")

print(
    f"\nBest validation RMSE: "
    f"{global_best_fitness:.6f}"
)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", filename)

print("\nTest data were not loaded or evaluated.")
print("Proceed to Step 6 for model training and evaluation.")

STEP 5: SWO HYPERPARAMETER OPTIMIZATION
Device: cpu

Training feature tensor: (28, 350, 10)
Training target tensor: (28, 350)
Validation feature tensor: (6, 350, 10)
Validation target tensor: (6, 350)
Adjacency matrix: (350, 350)

Initializing optimization population...
Initial candidate 1/8 | Validation RMSE = 0.328149 | Parameters = {'learning_rate': 0.0001, 'hidden_dim': 16, 'dropout': 0.3, 'window': 3, 'weight_decay': 1e-06}
Initial candidate 2/8 | Validation RMSE = 0.055882 | Parameters = {'learning_rate': 0.0003, 'hidden_dim': 64, 'dropout': 0.1, 'window': 2, 'weight_decay': 0.0001}
Initial candidate 3/8 | Validation RMSE = 0.099905 | Parameters = {'learning_rate': 0.001, 'hidden_dim': 16, 'dropout': 0.1, 'window': 2, 'weight_decay': 1e-06}
Initial candidate 4/8 | Validation RMSE = 0.108107 | Parameters = {'learning_rate': 0.0003, 'hidden_dim': 64, 'dropout': 0.1, 'window': 3, 'weight_decay': 0.0001}
Initial candidate 5/8 | Validation RMSE = 0.173507 | Parameters = {'learning_rat

In [11]:
import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

# ---------------------- 1. PATHS --------------------------------

BASE_DIR = r"G:\5287"

STEP4_DIR = os.path.join(
    BASE_DIR, "Step4_Spatial_Graph_Construction"
)

STEP5_DIR = os.path.join(
    BASE_DIR, "Step5_SWO_Optimization"
)

OUT_DIR = os.path.join(
    BASE_DIR, "Step6_STGCN_Model_Evaluation"
)

os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(
    STEP4_DIR, "training_graph_tensor.npz"
)

VAL_FILE = os.path.join(
    STEP4_DIR, "validation_graph_tensor.npz"
)

TEST_FILE = os.path.join(
    STEP4_DIR, "testing_graph_tensor.npz"
)

ADJ_FILE = os.path.join(
    STEP4_DIR, "normalized_adjacency_matrix.npy"
)

CONFIG_FILE = os.path.join(
    STEP5_DIR, "best_swo_hyperparameters.json"
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("STEP 6: SWO-STGCN MODEL EVALUATION")
print("=" * 70)

print("Device:", DEVICE)
print("Output:", OUT_DIR)


# ---------------------- 2. LOAD DATA ----------------------------

def load_npz(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(filepath)

    with np.load(filepath, allow_pickle=False) as data:
        result = {
            key: data[key]
            for key in data.files
        }

    print("\nLoaded:", os.path.basename(filepath))

    for key, value in result.items():
        print(f"{key}: {value.shape}")

    return result


train_data = load_npz(TRAIN_FILE)
val_data = load_npz(VAL_FILE)
test_data = load_npz(TEST_FILE)

adjacency = np.load(ADJ_FILE).astype(np.float32)

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = json.load(f)

# Your Step 5 configuration stores values under best_parameters.
best = config.get("best_parameters", config)

print("\nOptimized parameters:")
print(json.dumps(best, indent=4))


# ---------------------- 3. EXTRACT ARRAYS -----------------------

X_train = train_data["features"].astype(np.float32)
Y_train = train_data["target"].astype(np.float32)

X_val = val_data["features"].astype(np.float32)
Y_val = val_data["target"].astype(np.float32)

X_test = test_data["features"].astype(np.float32)
Y_test = test_data["target"].astype(np.float32)

train_mask = train_data["observation_mask"].astype(bool)
val_mask = val_data["observation_mask"].astype(bool)
test_mask = test_data["observation_mask"].astype(bool)

train_years = train_data["years"]
val_years = val_data["years"]
test_years = test_data["years"]

regions = test_data["regions"]

# Expected shapes:
# X: (years, regions, features)
# Y: (years, regions)
# Mask: (years, regions)

assert X_train.ndim == 3, X_train.shape
assert X_val.ndim == 3, X_val.shape
assert X_test.ndim == 3, X_test.shape

assert Y_train.shape == X_train.shape[:2]
assert Y_val.shape == X_val.shape[:2]
assert Y_test.shape == X_test.shape[:2]

assert adjacency.shape == (
    X_train.shape[1], X_train.shape[1]
)

assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]
assert X_train.shape[2] == X_val.shape[2] == X_test.shape[2]

print("\nVerified input shapes:")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)
print("Adjacency:", adjacency.shape)


# ---------------------- 4. HYPERPARAMETERS ----------------------

hidden_dim = int(best.get("hidden_dim", 32))
dropout_rate = float(best.get("dropout", 0.3))
learning_rate = float(best.get("learning_rate", 0.001))
weight_decay = float(best.get("weight_decay", 0.0001))
window = int(best.get("window", 2))

EPOCHS = 50
BATCH_SIZE = 8

print("\nModel parameters:")
print("Hidden dimension:", hidden_dim)
print("Dropout:", dropout_rate)
print("Learning rate:", learning_rate)
print("Weight decay:", weight_decay)
print("Temporal window:", window)


# ---------------------- 5. TARGET SCALING -----------------------

# Fit target scaling using observed training targets only.
valid_train = (
    train_mask
    & np.isfinite(Y_train)
)

if not np.any(valid_train):
    raise ValueError("No valid training targets found.")

target_mean = float(Y_train[valid_train].mean())
target_std = float(Y_train[valid_train].std())

if target_std < 1e-8:
    target_std = 1.0

print("\nTraining target mean:", target_mean)
print("Training target std:", target_std)


def scale_target(y):
    return (y - target_mean) / target_std


def inverse_target(y):
    return y * target_std + target_mean


# ---------------------- 6. CREATE SEQUENCES ---------------------

def create_sequences(
    X,
    Y,
    mask,
    years,
    window,
    previous_X=None,
    previous_Y=None,
    previous_mask=None,
    previous_years=None
):
    """
    Each sample uses the preceding 'window' years of features
    to predict GTFP in the next year.

    Validation/test sequences may use historical features from
    earlier splits. No validation/test target is used as input.
    """

    if previous_X is not None:
        X_all = np.concatenate([previous_X, X], axis=0)
        Y_all = np.concatenate([previous_Y, Y], axis=0)
        mask_all = np.concatenate(
            [previous_mask, mask], axis=0
        )
        years_all = np.concatenate(
            [previous_years, years], axis=0
        )
        offset = len(previous_X)
    else:
        X_all = X
        Y_all = Y
        mask_all = mask
        years_all = years
        offset = 0

    seq_X, seq_Y, seq_mask, seq_year = [], [], [], []

    # Predict the target at index t using features t-window:t.
    for t in range(window, len(X_all)):
        if t < offset:
            continue

        x_window = X_all[t-window:t]
        target = Y_all[t]
        target_mask = (
            mask_all[t] & np.isfinite(target)
        )

        seq_X.append(x_window)
        seq_Y.append(target)
        seq_mask.append(target_mask)
        seq_year.append(years_all[t])

    if len(seq_X) == 0:
        return (
            np.empty(
                (0, window, X.shape[1], X.shape[2]),
                dtype=np.float32
            ),
            np.empty((0, X.shape[1]), dtype=np.float32),
            np.empty((0, X.shape[1]), dtype=bool),
            np.empty((0,), dtype=years.dtype)
        )

    return (
        np.asarray(seq_X, dtype=np.float32),
        np.asarray(seq_Y, dtype=np.float32),
        np.asarray(seq_mask, dtype=bool),
        np.asarray(seq_year)
    )


Xtr, Ytr, Mtr, years_tr = create_sequences(
    X_train, Y_train, train_mask, train_years, window
)

Xva, Yva, Mva, years_va = create_sequences(
    X_val,
    Y_val,
    val_mask,
    val_years,
    window,
    previous_X=X_train[-window:],
    previous_Y=Y_train[-window:],
    previous_mask=train_mask[-window:],
    previous_years=train_years[-window:]
)

# Test predictions use training and validation features as history.
X_history = np.concatenate(
    [X_train, X_val], axis=0
)[-window:]

Y_history = np.concatenate(
    [Y_train, Y_val], axis=0
)[-window:]

M_history = np.concatenate(
    [train_mask, val_mask], axis=0
)[-window:]

years_history = np.concatenate(
    [train_years, val_years], axis=0
)[-window:]

Xte, Yte, Mte, years_te = create_sequences(
    X_test,
    Y_test,
    test_mask,
    test_years,
    window,
    previous_X=X_history,
    previous_Y=Y_history,
    previous_mask=M_history,
    previous_years=years_history
)

print("\nSequence shapes:")
print("Training:", Xtr.shape, Ytr.shape)
print("Validation:", Xva.shape, Yva.shape)
print("Testing:", Xte.shape, Yte.shape)

if len(Xtr) == 0 or len(Xva) == 0 or len(Xte) == 0:
    raise ValueError(
        "No sequences were created. Check the temporal window "
        "and the number of years in each split."
    )


# ---------------------- 7. SCALE TARGETS ------------------------

Ytr_scaled = scale_target(Ytr)
Yva_scaled = scale_target(Yva)
Yte_scaled = scale_target(Yte)


# ---------------------- 8. DEFINE MODEL -------------------------

class GraphConv(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, adj):
        # x: batch, time, nodes, features
        x = torch.einsum("ij,btjf->btif", adj, x)
        return self.linear(x)


class SWO_STGCN(nn.Module):
    def __init__(
        self,
        input_features,
        hidden_dim,
        dropout
    ):
        super().__init__()

        self.graph_conv = GraphConv(
            input_features, hidden_dim
        )

        self.temporal_conv = nn.Conv2d(
            hidden_dim,
            hidden_dim,
            kernel_size=(3, 1),
            padding=(1, 0)
        )

        self.dropout = nn.Dropout(dropout)
        self.output_layer = nn.Linear(hidden_dim, 1)
        self.activation = nn.ReLU()

    def forward(self, x, adj):
        # x: batch, time, nodes, features

        x = self.activation(
            self.graph_conv(x, adj)
        )

        # batch, channels, time, nodes
        x = x.permute(0, 3, 1, 2)

        x = self.activation(
            self.temporal_conv(x)
        )

        # Last temporal position
        x = x[:, :, -1, :]

        # batch, nodes, channels
        x = x.permute(0, 2, 1)
        x = self.dropout(x)

        # batch, nodes
        return self.output_layer(x).squeeze(-1)


model = SWO_STGCN(
    input_features=X_train.shape[-1],
    hidden_dim=hidden_dim,
    dropout=dropout_rate
).to(DEVICE)

adj_tensor = torch.tensor(
    adjacency, dtype=torch.float32, device=DEVICE
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

criterion = nn.MSELoss()

print("\nTrainable parameters:",
      sum(p.numel() for p in model.parameters()
          if p.requires_grad))


# ---------------------- 9. TENSOR CONVERSION --------------------

def to_device(array, dtype=torch.float32):
    return torch.tensor(
        array, dtype=dtype, device=DEVICE
    )


Xtr_t = to_device(Xtr)
Ytr_t = to_device(Ytr_scaled)
Mtr_t = to_device(Mtr, dtype=torch.bool)

Xva_t = to_device(Xva)
Yva_t = to_device(Yva_scaled)
Mva_t = to_device(Mva, dtype=torch.bool)

Xte_t = to_device(Xte)
Yte_t = to_device(Yte_scaled)
Mte_t = to_device(Mte, dtype=torch.bool)


# ---------------------- 10. TRAIN MODEL -------------------------

history = []

print("\nTraining compatible STGCN model...")

for epoch in range(EPOCHS):
    model.train()

    permutation = torch.randperm(
        len(Xtr_t), device=DEVICE
    )

    batch_losses = []

    for start in range(0, len(Xtr_t), BATCH_SIZE):
        idx = permutation[start:start + BATCH_SIZE]

        xb = Xtr_t[idx]
        yb = Ytr_t[idx]
        mb = Mtr_t[idx]

        optimizer.zero_grad()

        prediction = model(xb, adj_tensor)

        if not torch.any(mb):
            continue

        loss = ((prediction - yb)[mb] ** 2).mean()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        optimizer.step()
        batch_losses.append(loss.item())

    train_loss = (
        float(np.mean(batch_losses))
        if batch_losses else np.nan
    )

    model.eval()

    with torch.no_grad():
        val_prediction = model(Xva_t, adj_tensor)

        if torch.any(Mva_t):
            val_loss = (
                (val_prediction - Yva_t)[Mva_t] ** 2
            ).mean().item()
        else:
            val_loss = np.nan

    history.append({
        "Epoch": epoch + 1,
        "Training_MSE": train_loss,
        "Validation_MSE": val_loss
    })

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1:03d}/{EPOCHS} | "
            f"Train MSE: {train_loss:.6f} | "
            f"Validation MSE: {val_loss:.6f}"
        )

# Save the trained model.
torch.save(
    model.state_dict(),
    os.path.join(OUT_DIR, "step6_stw_stgcn_model.pth")
)

history_df = pd.DataFrame(history)

history_df.to_csv(
    os.path.join(OUT_DIR, "training_history.csv"),
    index=False
)


# ---------------------- 11. TRAINING CURVE ----------------------

plt.figure(figsize=(9, 5))

plt.plot(
    history_df["Epoch"],
    history_df["Training_MSE"],
    marker="o",
    markersize=3,
    label="Training MSE"
)

plt.plot(
    history_df["Epoch"],
    history_df["Validation_MSE"],
    marker="s",
    markersize=3,
    label="Validation MSE"
)

plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("SWO-STGCN Training and Validation Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()

plt.savefig(
    os.path.join(OUT_DIR, "training_validation_loss.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ---------------------- 12. PREDICTION --------------------------

model.eval()

with torch.no_grad():
    train_pred_scaled = model(
        Xtr_t, adj_tensor
    ).cpu().numpy()

    val_pred_scaled = model(
        Xva_t, adj_tensor
    ).cpu().numpy()

    test_pred_scaled = model(
        Xte_t, adj_tensor
    ).cpu().numpy()

train_pred = inverse_target(train_pred_scaled)
val_pred = inverse_target(val_pred_scaled)
test_pred = inverse_target(test_pred_scaled)

actual_train = Ytr
actual_val = Yva
actual_test = Yte


# ---------------------- 13. METRICS ------------------------------

def calculate_metrics(actual, predicted, mask, name):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    mask = np.asarray(mask, dtype=bool)

    valid = (
        mask
        & np.isfinite(actual)
        & np.isfinite(predicted)
    )

    y_true = actual[valid]
    y_pred = predicted[valid]

    if len(y_true) == 0:
        raise ValueError(
            f"No valid predictions available for {name}."
        )

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = (
        r2_score(y_true, y_pred)
        if len(y_true) > 1 and np.var(y_true) > 0
        else np.nan
    )

    nonzero = np.abs(y_true) > 1e-8

    mape = (
        np.mean(
            np.abs(
                (y_true[nonzero] - y_pred[nonzero])
                / y_true[nonzero]
            )
        ) * 100
        if np.any(nonzero)
        else np.nan
    )

    return {
        "Dataset": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE_Percent": mape,
        "Valid_Observations": len(y_true)
    }


metrics = pd.DataFrame([
    calculate_metrics(
        actual_train, train_pred, Mtr, "Training"
    ),
    calculate_metrics(
        actual_val, val_pred, Mva, "Validation"
    ),
    calculate_metrics(
        actual_test, test_pred, Mte, "Testing"
    )
])

metrics.to_csv(
    os.path.join(OUT_DIR, "performance_metrics.csv"),
    index=False
)

print("\nPerformance metrics:")
print(metrics.to_string(index=False))


# ---------------------- 14. PREDICTION CSV ----------------------

def prediction_dataframe(actual, predicted, mask, years, split):
    rows = []

    for t in range(len(years)):
        for r in range(actual.shape[1]):
            if not mask[t, r]:
                continue

            if not (
                np.isfinite(actual[t, r])
                and np.isfinite(predicted[t, r])
            ):
                continue

            rows.append({
                "Dataset": split,
                "Year": years[t],
                "Region": regions[r],
                "Actual_GTFP": actual[t, r],
                "Predicted_GTFP": predicted[t, r],
                "Prediction_Error": (
                    predicted[t, r] - actual[t, r]
                ),
                "Absolute_Error": abs(
                    predicted[t, r] - actual[t, r]
                )
            })

    return pd.DataFrame(rows)


predictions = pd.concat([
    prediction_dataframe(
        actual_train, train_pred, Mtr, years_tr, "Training"
    ),
    prediction_dataframe(
        actual_val, val_pred, Mva, years_va, "Validation"
    ),
    prediction_dataframe(
        actual_test, test_pred, Mte, years_te, "Testing"
    )
], ignore_index=True)

predictions.to_csv(
    os.path.join(OUT_DIR, "all_prediction_results.csv"),
    index=False
)

predictions[
    predictions["Dataset"] == "Testing"
].to_csv(
    os.path.join(OUT_DIR, "test_prediction_results.csv"),
    index=False
)


# ---------------------- 15. ACTUAL VS PREDICTED ------------------

valid = (
    Mte
    & np.isfinite(actual_test)
    & np.isfinite(test_pred)
)

y_true = actual_test[valid]
y_pred = test_pred[valid]

plt.figure(figsize=(8, 7))

plt.scatter(
    y_true, y_pred,
    alpha=0.55,
    s=20
)

lower = min(y_true.min(), y_pred.min())
upper = max(y_true.max(), y_pred.max())

plt.plot(
    [lower, upper],
    [lower, upper],
    linestyle="--",
    linewidth=2,
    label="Ideal Prediction"
)

plt.xlabel("Actual GTFP")
plt.ylabel("Predicted GTFP")
plt.title("Actual vs. Predicted GTFP — Test Set")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.35)
plt.tight_layout()

plt.savefig(
    os.path.join(OUT_DIR, "actual_vs_predicted.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ---------------------- 16. ERROR DISTRIBUTION ------------------

errors = y_pred - y_true

plt.figure(figsize=(9, 5))

plt.hist(
    errors,
    bins=35,
    edgecolor="black",
    alpha=0.8
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=2,
    label="Zero Error"
)

plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.title("SWO-STGCN Test Prediction Error Distribution")
plt.legend()
plt.grid(True, axis="y", linestyle=":", alpha=0.35)
plt.tight_layout()

plt.savefig(
    os.path.join(OUT_DIR, "prediction_error_distribution.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ---------------------- 17. REGIONAL PERFORMANCE ----------------

regional_rows = []

for r in range(actual_test.shape[1]):
    valid_r = (
        Mte[:, r]
        & np.isfinite(actual_test[:, r])
        & np.isfinite(test_pred[:, r])
    )

    if not np.any(valid_r):
        continue

    yr = actual_test[valid_r, r]
    pr = test_pred[valid_r, r]

    regional_rows.append({
        "Region": regions[r],
        "MAE": mean_absolute_error(yr, pr),
        "RMSE": np.sqrt(mean_squared_error(yr, pr)),
        "R2": (
            r2_score(yr, pr)
            if len(yr) > 1 and np.var(yr) > 0
            else np.nan
        ),
        "Observations": len(yr)
    })

regional_df = pd.DataFrame(regional_rows)

regional_df.to_csv(
    os.path.join(OUT_DIR, "regional_performance.csv"),
    index=False
)

plt.figure(figsize=(9, 5))

plt.hist(
    regional_df["RMSE"].dropna(),
    bins=25,
    edgecolor="black",
    alpha=0.8
)

plt.xlabel("Regional RMSE")
plt.ylabel("Number of Regions")
plt.title("Regional Prediction Error Distribution")
plt.grid(True, axis="y", linestyle=":", alpha=0.35)
plt.tight_layout()

plt.savefig(
    os.path.join(OUT_DIR, "regional_rmse_distribution.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ---------------------- 18. SAVE SUMMARY ------------------------

summary = {
    "Model": "SWO-STGCN",
    "Device": str(DEVICE),
    "Training_Years": train_years.tolist(),
    "Validation_Years": val_years.tolist(),
    "Testing_Years": test_years.tolist(),
    "Window_Size": window,
    "Regions": int(X_train.shape[1]),
    "KPCA_Features": int(X_train.shape[2]),
    "Target_Mean_Training": target_mean,
    "Target_Std_Training": target_std,
    "Optimized_Hyperparameters": best,
    "Metrics": metrics.to_dict(orient="records")
}

with open(
    os.path.join(OUT_DIR, "evaluation_summary.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("\n" + "=" * 70)
print("STEP 6 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nResults saved to:")
print(OUT_DIR)

print("\nGenerated files:")
for filename in sorted(os.listdir(OUT_DIR)):
    print(filename)

STEP 6: SWO-STGCN MODEL EVALUATION
Device: cpu
Output: G:*87\Step6_STGCN_Model_Evaluation

Loaded: training_graph_tensor.npz
features: (28, 350, 10)
observation_mask: (28, 350)
years: (28,)
regions: (350,)
feature_names: (10,)
target: (28, 350)

Loaded: validation_graph_tensor.npz
features: (6, 350, 10)
observation_mask: (6, 350)
years: (6,)
regions: (350,)
feature_names: (10,)
target: (6, 350)

Loaded: testing_graph_tensor.npz
features: (6, 350, 10)
observation_mask: (6, 350)
years: (6,)
regions: (350,)
feature_names: (10,)
target: (6, 350)

Optimized parameters:
{
    "learning_rate": 0.001,
    "hidden_dim": 32,
    "dropout": 0.3,
    "window": 2,
    "weight_decay": 0.0001
}

Verified input shapes:
Training: (28, 350, 10)
Validation: (6, 350, 10)
Testing: (6, 350, 10)
Adjacency: (350, 350)

Model parameters:
Hidden dimension: 32
Dropout: 0.3
Learning rate: 0.001
Weight decay: 0.0001
Temporal window: 2

Training target mean: 0.3845180571079254
Training target std: 0.127084285020828

# FIVE-FOLD CROSS-VALIDATION

In [17]:
import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from scipy.stats import ttest_rel, wilcoxon

warnings.filterwarnings("ignore")

# ------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------

BASE_DIR = r"G:\5287"

STEP4_DIR = os.path.join(
    BASE_DIR, "Step4_Spatial_Graph_Construction"
)

STEP5_DIR = os.path.join(
    BASE_DIR, "Step5_SWO_Optimization"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR, "Step7_Five_Fold_Cross_Validation"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(
    STEP4_DIR, "training_graph_tensor.npz"
)

VAL_FILE = os.path.join(
    STEP4_DIR, "validation_graph_tensor.npz"
)

TEST_FILE = os.path.join(
    STEP4_DIR, "testing_graph_tensor.npz"
)

CONFIG_FILE = os.path.join(
    STEP5_DIR, "best_swo_hyperparameters.json"
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 75)
print("FIVE-FOLD CROSS-VALIDATION: SWO-STGCN VS STGNN")
print("=" * 75)
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR)


# ------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------

def load_npz(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"Missing input file: {filepath}"
        )

    with np.load(filepath, allow_pickle=False) as data:
        return {key: data[key] for key in data.files}


train_data = load_npz(TRAIN_FILE)
val_data = load_npz(VAL_FILE)
test_data = load_npz(TEST_FILE)

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    saved_config = json.load(f)

# Step 5 stores optimized parameters under best_parameters.
params = saved_config.get(
    "best_parameters", saved_config
)

# Combine chronologically ordered data splits.
X = np.concatenate([
    train_data["features"],
    val_data["features"],
    test_data["features"]
], axis=0).astype(np.float32)

Y = np.concatenate([
    train_data["target"],
    val_data["target"],
    test_data["target"]
], axis=0).astype(np.float32)

MASK = np.concatenate([
    train_data["observation_mask"],
    val_data["observation_mask"],
    test_data["observation_mask"]
], axis=0).astype(bool)

YEARS = np.concatenate([
    train_data["years"],
    val_data["years"],
    test_data["years"]
])

REGIONS = train_data["regions"]

# Sort all arrays by year to guarantee chronological order.
order = np.argsort(YEARS)

X = X[order]
Y = Y[order]
MASK = MASK[order]
YEARS = YEARS[order]

if len(np.unique(YEARS)) != len(YEARS):
    raise ValueError(
        "Duplicate years detected. Verify that the input splits "
        "do not contain overlapping years."
    )

if not np.all(np.diff(YEARS.astype(float)) > 0):
    raise ValueError(
        "Years must be strictly increasing."
    )

n_years, n_regions, n_features = X.shape

print("\nCombined data:")
print("Years:", YEARS)
print("Feature tensor:", X.shape)
print("Target tensor:", Y.shape)
print("Regions:", n_regions)
print("Features:", n_features)

if n_years < 20:
    raise ValueError(
        "Too few annual observations for this five-fold design."
    )


# ------------------------------------------------
# 3. MODEL HYPERPARAMETERS
# ------------------------------------------------

HIDDEN_DIM = int(params.get("hidden_dim", 32))
DROPOUT = float(params.get("dropout", 0.3))
LEARNING_RATE = float(params.get("learning_rate", 0.001))
WEIGHT_DECAY = float(params.get("weight_decay", 0.0001))
WINDOW = int(params.get("window", 2))

EPOCHS = 35
BATCH_SIZE = 4

print("\nHyperparameters:")
print("Hidden dimension:", HIDDEN_DIM)
print("Dropout:", DROPOUT)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Temporal window:", WINDOW)
print("Epochs per fold:", EPOCHS)


# ------------------------------------------------
# 4. FIVE CHRONOLOGICAL FOLDS
# ------------------------------------------------

INITIAL_TRAIN_YEARS = n_years // 2
VALIDATION_BLOCK = (n_years - INITIAL_TRAIN_YEARS) // 5

if VALIDATION_BLOCK < 1:
    raise ValueError(
        "Insufficient years to construct five validation folds."
    )

fold_definitions = []

for fold_idx in range(5):
    train_end = INITIAL_TRAIN_YEARS + (
        fold_idx * VALIDATION_BLOCK
    )

    val_start = train_end
    val_end = (
        INITIAL_TRAIN_YEARS
        + (fold_idx + 1) * VALIDATION_BLOCK
    )

    if fold_idx == 4:
        val_end = n_years

    if val_end <= val_start:
        raise ValueError(
            f"Invalid validation range in fold {fold_idx + 1}"
        )

    fold_definitions.append({
        "fold": fold_idx + 1,
        "train_end": train_end,
        "val_start": val_start,
        "val_end": val_end
    })

print("\nChronological fold definitions:")

for fold in fold_definitions:
    print(
        f"Fold {fold['fold']}: "
        f"Train {YEARS[0]}-{YEARS[fold['train_end'] - 1]}, "
        f"Validate {YEARS[fold['val_start']]}-"
        f"{YEARS[fold['val_end'] - 1]}"
    )


# ------------------------------------------------
# 5. GRAPH CONSTRUCTION
# ------------------------------------------------

def build_training_graph(training_features, top_k=5):
    """
    Construct a symmetric feature-similarity graph using
    the training period only.

    This avoids using validation-period features to estimate
    graph similarities.
    """

    # Mean regional representation from training years.
    regional_features = np.mean(
        training_features, axis=0
    )

    # Standardize using regional feature statistics.
    feature_mean = regional_features.mean(axis=0)
    feature_std = regional_features.std(axis=0)
    feature_std[feature_std < 1e-8] = 1.0

    regional_features = (
        regional_features - feature_mean
    ) / feature_std

    # Pairwise squared Euclidean distances.
    squared_norm = np.sum(
        regional_features ** 2, axis=1, keepdims=True
    )

    distance_sq = (
        squared_norm
        + squared_norm.T
        - 2 * regional_features @ regional_features.T
    )

    distance_sq = np.maximum(distance_sq, 0.0)

    positive = distance_sq[
        np.triu_indices_from(distance_sq, k=1)
    ]

    positive = positive[positive > 0]

    bandwidth = (
        np.sqrt(np.median(positive))
        if len(positive) > 0
        else 1.0
    )

    bandwidth = max(float(bandwidth), 1e-8)

    similarity = np.exp(
        -distance_sq / (2 * bandwidth ** 2)
    )

    np.fill_diagonal(similarity, 0.0)

    n_nodes = similarity.shape[0]
    adjacency = np.zeros_like(similarity)

    k = min(top_k, n_nodes - 1)

    for i in range(n_nodes):
        neighbors = np.argsort(
            similarity[i]
        )[-k:]

        adjacency[i, neighbors] = similarity[
            i, neighbors
        ]

    # Symmetric graph.
    adjacency = np.maximum(
        adjacency, adjacency.T
    )

    # Add self-loops.
    adjacency += np.eye(n_nodes, dtype=np.float32)

    # Symmetric normalized adjacency.
    degree = adjacency.sum(axis=1)
    degree[degree <= 0] = 1.0

    d_inv_sqrt = np.diag(
        1.0 / np.sqrt(degree)
    )

    normalized = (
        d_inv_sqrt @ adjacency @ d_inv_sqrt
    )

    return normalized.astype(np.float32)


# ------------------------------------------------
# 6. SEQUENCE CREATION
# ------------------------------------------------

def create_fold_sequences(
    X,
    Y,
    MASK,
    train_end,
    val_end,
    window
):

    X_train_seq, Y_train_seq, M_train_seq = [], [], []
    X_val_seq, Y_val_seq, M_val_seq = [], [], []

    for t in range(window, train_end):
        x_seq = X[t-window:t]
        y_target = Y[t]
        target_mask = (
            MASK[t] & np.isfinite(y_target)
        )

        if np.any(target_mask):
            X_train_seq.append(x_seq)
            Y_train_seq.append(y_target)
            M_train_seq.append(target_mask)

    for t in range(max(window, train_end), val_end):
        x_seq = X[t-window:t]
        y_target = Y[t]
        target_mask = (
            MASK[t] & np.isfinite(y_target)
        )

        if np.any(target_mask):
            X_val_seq.append(x_seq)
            Y_val_seq.append(y_target)
            M_val_seq.append(target_mask)

    if not X_train_seq or not X_val_seq:
        raise ValueError(
            "Empty sequence collection. Check fold boundaries "
            "and temporal window."
        )

    return (
        np.asarray(X_train_seq, dtype=np.float32),
        np.asarray(Y_train_seq, dtype=np.float32),
        np.asarray(M_train_seq, dtype=bool),
        np.asarray(X_val_seq, dtype=np.float32),
        np.asarray(Y_val_seq, dtype=np.float32),
        np.asarray(M_val_seq, dtype=bool)
    )


# ------------------------------------------------
# 7. DEFINE MODEL ARCHITECTURES
# ------------------------------------------------

class GraphConvolution(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, hidden_dim)

    def forward(self, x, adjacency):
        # x: batch, time, nodes, features
        x = torch.einsum(
            "ij,btjf->btif", adjacency, x
        )
        return self.linear(x)


class SWOSTGCN(nn.Module):
    """
    Spatial graph convolution followed by temporal convolution.
    """

    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()

        self.graph = GraphConvolution(
            input_dim, hidden_dim
        )

        self.temporal = nn.Conv2d(
            hidden_dim,
            hidden_dim,
            kernel_size=(3, 1),
            padding=(1, 0)
        )

        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x, adjacency):
        x = self.activation(
            self.graph(x, adjacency)
        )

        # batch, channels, time, nodes
        x = x.permute(0, 3, 1, 2)

        x = self.activation(
            self.temporal(x)
        )

        # Last temporal representation.
        x = x[:, :, -1, :]

        # batch, nodes, channels
        x = x.permute(0, 2, 1)

        x = self.dropout(x)

        return self.output(x).squeeze(-1)


class STGNN(nn.Module):
    """
    Baseline STGNN:
    graph convolution followed by a GRU temporal encoder.
    """

    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()

        self.graph = GraphConvolution(
            input_dim, hidden_dim
        )

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, 1)
        self.activation = nn.ReLU()

    def forward(self, x, adjacency):
        x = self.activation(
            self.graph(x, adjacency)
        )

        # x: batch, time, nodes, hidden
        batch, steps, nodes, hidden = x.shape

        # Process each node's temporal sequence.
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(
            batch * nodes, steps, hidden
        )

        _, h = self.gru(x)

        # Last GRU hidden state.
        x = h[-1]

        x = self.dropout(x)
        x = self.output(x)

        return x.reshape(batch, nodes)


# ------------------------------------------------
# 8. METRICS
# ------------------------------------------------

def calculate_metrics(y_true, y_pred, mask):
    valid = (
        mask
        & np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    true_values = y_true[valid]
    pred_values = y_pred[valid]

    if len(true_values) == 0:
        raise ValueError(
            "No valid targets available for metric calculation."
        )

    mae = mean_absolute_error(
        true_values, pred_values
    )

    rmse = np.sqrt(
        mean_squared_error(
            true_values, pred_values
        )
    )

    r2 = (
        r2_score(true_values, pred_values)
        if len(true_values) > 1
        and np.var(true_values) > 0
        else np.nan
    )

    return mae, rmse, r2


# ------------------------------------------------
# 9. MODEL TRAINING AND EVALUATION
# ------------------------------------------------

def train_and_evaluate(
    model_class,
    X_train_seq,
    Y_train_seq,
    M_train_seq,
    X_val_seq,
    Y_val_seq,
    M_val_seq,
    adjacency,
    fold_seed
):
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)

    # Target scaling fitted only on observed training targets.
    train_targets = Y_train_seq[M_train_seq]

    target_mean = float(train_targets.mean())
    target_std = float(train_targets.std())

    if target_std < 1e-8:
        target_std = 1.0

    Y_train_scaled = (
        Y_train_seq - target_mean
    ) / target_std

    X_train_t = torch.tensor(
        X_train_seq, dtype=torch.float32, device=DEVICE
    )

    Y_train_t = torch.tensor(
        Y_train_scaled, dtype=torch.float32, device=DEVICE
    )

    M_train_t = torch.tensor(
        M_train_seq, dtype=torch.bool, device=DEVICE
    )

    X_val_t = torch.tensor(
        X_val_seq, dtype=torch.float32, device=DEVICE
    )

    adjacency_t = torch.tensor(
        adjacency, dtype=torch.float32, device=DEVICE
    )

    model = model_class(
        input_dim=n_features,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    best_state = None
    best_val_loss = float("inf")
    patience = 7
    no_improve = 0

    for epoch in range(EPOCHS):
        model.train()

        permutation = torch.randperm(
            len(X_train_t), device=DEVICE
        )

        for start in range(
            0, len(X_train_t), BATCH_SIZE
        ):
            idx = permutation[
                start:start + BATCH_SIZE
            ]

            xb = X_train_t[idx]
            yb = Y_train_t[idx]
            mb = M_train_t[idx]

            optimizer.zero_grad()

            pred = model(xb, adjacency_t)

            if not torch.any(mb):
                continue

            loss = (
                (pred - yb)[mb] ** 2
            ).mean()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            optimizer.step()

        # Early stopping uses only the fold's validation period.
        model.eval()

        with torch.no_grad():
            val_pred_scaled = model(
                X_val_t, adjacency_t
            ).cpu().numpy()

        val_pred_original = (
            val_pred_scaled * target_std + target_mean
        )

        valid = (
            M_val_seq
            & np.isfinite(Y_val_seq)
            & np.isfinite(val_pred_original)
        )

        if not np.any(valid):
            raise ValueError(
                "No valid validation targets in this fold."
            )

        val_loss = np.mean(
            (
                val_pred_original[valid]
                - Y_val_seq[valid]
            ) ** 2
        )

        if val_loss < best_val_loss:
            best_val_loss = float(val_loss)
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()

    with torch.no_grad():
        val_pred_scaled = model(
            X_val_t, adjacency_t
        ).cpu().numpy()

    val_pred_original = (
        val_pred_scaled * target_std + target_mean
    )

    return calculate_metrics(
        Y_val_seq,
        val_pred_original,
        M_val_seq
    )


# ------------------------------------------------
# 10. RUN FIVE-FOLD CROSS-VALIDATION
# ------------------------------------------------

results = []

for fold in fold_definitions:
    fold_number = fold["fold"]

    print("\n" + "-" * 70)
    print(f"PROCESSING FOLD {fold_number}")
    print("-" * 70)

    train_end = fold["train_end"]
    val_start = fold["val_start"]
    val_end = fold["val_end"]

    # Construct the graph using the current fold's training
    # period only.
    fold_adjacency = build_training_graph(
        X[:train_end],
        top_k=5
    )

    (
        Xtr, Ytr, Mtr,
        Xva, Yva, Mva
    ) = create_fold_sequences(
        X=X,
        Y=Y,
        MASK=MASK,
        train_end=train_end,
        val_end=val_end,
        window=WINDOW
    )

    print(
        "Training sequences:", Xtr.shape,
        "| Validation sequences:", Xva.shape
    )

    print(
        "Training years:",
        YEARS[0], "to", YEARS[train_end - 1]
    )

    print(
        "Validation years:",
        YEARS[val_start], "to", YEARS[val_end - 1]
    )

    # Proposed SWO-STGCN.
    swo_mae, swo_rmse, swo_r2 = train_and_evaluate(
        SWOSTGCN,
        Xtr, Ytr, Mtr,
        Xva, Yva, Mva,
        fold_adjacency,
        SEED + fold_number
    )

    # Baseline STGNN.
    stgnn_mae, stgnn_rmse, stgnn_r2 = train_and_evaluate(
        STGNN,
        Xtr, Ytr, Mtr,
        Xva, Yva, Mva,
        fold_adjacency,
        SEED + fold_number
    )

    results.append({
        "Fold": f"Fold {fold_number}",
        "SWO-STGCN MAE": swo_mae,
        "SWO-STGCN RMSE": swo_rmse,
        "SWO-STGCN R2": swo_r2,
        "STGNN MAE": stgnn_mae,
        "STGNN RMSE": stgnn_rmse,
        "STGNN R2": stgnn_r2,
        "Training_Start_Year": YEARS[0],
        "Training_End_Year": YEARS[train_end - 1],
        "Validation_Start_Year": YEARS[val_start],
        "Validation_End_Year": YEARS[val_end - 1]
    })

    print(
        f"SWO-STGCN: MAE={swo_mae:.6f}, "
        f"RMSE={swo_rmse:.6f}, R2={swo_r2:.6f}"
    )

    print(
        f"STGNN: MAE={stgnn_mae:.6f}, "
        f"RMSE={stgnn_rmse:.6f}, R2={stgnn_r2:.6f}"
    )


# ------------------------------------------------
# 11. SAVE FIVE-FOLD TABLE
# ------------------------------------------------

results_df = pd.DataFrame(results)

metric_columns = [
    "SWO-STGCN MAE",
    "SWO-STGCN RMSE",
    "SWO-STGCN R2",
    "STGNN MAE",
    "STGNN RMSE",
    "STGNN R2"
]

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "five_fold_cross_validation_raw.csv"
    ),
    index=False
)

# Construct publication-style table.
table_rows = []

for _, row in results_df.iterrows():
    table_rows.append({
        "Fold": row["Fold"],
        "SWO-STGCN MAE": row["SWO-STGCN MAE"],
        "SWO-STGCN RMSE": row["SWO-STGCN RMSE"],
        "SWO-STGCN R2": row["SWO-STGCN R2"],
        "STGNN MAE": row["STGNN MAE"],
        "STGNN RMSE": row["STGNN RMSE"],
        "STGNN R2": row["STGNN R2"]
    })

summary_row = {"Fold": "Mean ± SD"}

for column in metric_columns:
    mean_value = results_df[column].mean()
    std_value = results_df[column].std(ddof=1)

    summary_row[column] = (
        f"{mean_value:.6f} ± {std_value:.6f}"
    )

table_rows.append(summary_row)

publication_table = pd.DataFrame(table_rows)

publication_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "five_fold_publication_table.csv"
    ),
    index=False
)

print("\n" + "=" * 90)
print("FIVE-FOLD CROSS-VALIDATION PERFORMANCE")
print("=" * 90)

print(publication_table.to_string(index=False))


# ------------------------------------------------
# 12. PAIRED STATISTICAL TESTS
# ------------------------------------------------
# The tests compare fold-level metric values.
# For MAE and RMSE, lower values indicate better performance.
# For R2, higher values indicate better performance.

statistical_results = []

comparisons = [
    ("MAE", "SWO-STGCN MAE", "STGNN MAE", "lower"),
    ("RMSE", "SWO-STGCN RMSE", "STGNN RMSE", "lower"),
    ("R2", "SWO-STGCN R2", "STGNN R2", "higher")
]

for metric_name, proposed_col, baseline_col, direction in comparisons:
    proposed = results_df[proposed_col].to_numpy(dtype=float)
    baseline = results_df[baseline_col].to_numpy(dtype=float)

    valid = np.isfinite(proposed) & np.isfinite(baseline)

    proposed = proposed[valid]
    baseline = baseline[valid]

    if len(proposed) < 2:
        t_stat, t_p = np.nan, np.nan
        w_stat, w_p = np.nan, np.nan
    else:
        t_result = ttest_rel(proposed, baseline)
        t_stat = float(t_result.statistic)
        t_p = float(t_result.pvalue)

        differences = proposed - baseline

        if np.allclose(differences, 0):
            w_stat, w_p = 0.0, 1.0
        else:
            try:
                w_result = wilcoxon(
                    proposed,
                    baseline,
                    alternative="two-sided",
                    method="auto"
                )
                w_stat = float(w_result.statistic)
                w_p = float(w_result.pvalue)
            except ValueError:
                w_stat, w_p = np.nan, np.nan

    mean_proposed = float(np.mean(proposed))
    mean_baseline = float(np.mean(baseline))

    if direction == "lower":
        improvement = (
            (mean_baseline - mean_proposed)
            / abs(mean_baseline) * 100
            if abs(mean_baseline) > 1e-12
            else np.nan
        )
    else:
        improvement = (
            (mean_proposed - mean_baseline)
            / abs(mean_baseline) * 100
            if abs(mean_baseline) > 1e-12
            else np.nan
        )

    statistical_results.append({
        "Metric": metric_name,
        "SWO-STGCN_Mean": mean_proposed,
        "STGNN_Mean": mean_baseline,
        "Difference_SWOSTGCN_minus_STGNN":
            mean_proposed - mean_baseline,
        "Improvement_Percent": improvement,
        "Paired_t_statistic": t_stat,
        "Paired_t_p_value": t_p,
        "Wilcoxon_statistic": w_stat,
        "Wilcoxon_p_value": w_p,
        "Significance_at_0.05": (
            "Yes" if np.isfinite(t_p) and t_p < 0.05
            else "No"
        )
    })

statistics_df = pd.DataFrame(statistical_results)

statistics_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "paired_statistical_comparison.csv"
    ),
    index=False
)

print("\n" + "=" * 90)
print("PAIRED STATISTICAL COMPARISON")
print("=" * 90)

print(statistics_df.to_string(index=False))


# ------------------------------------------------
# 13. VISUALIZATION
# ------------------------------------------------

# MAE comparison across folds.
plt.figure(figsize=(9, 5))

plt.plot(
    results_df["Fold"],
    results_df["SWO-STGCN MAE"],
    marker="o",
    linewidth=2,
    label="SWO-STGCN"
)

plt.plot(
    results_df["Fold"],
    results_df["STGNN MAE"],
    marker="s",
    linewidth=2,
    label="STGNN"
)

plt.xlabel("Cross-Validation Fold")
plt.ylabel("MAE")
plt.title("Five-Fold MAE Comparison")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "five_fold_mae_comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# RMSE comparison across folds.
plt.figure(figsize=(9, 5))

plt.plot(
    results_df["Fold"],
    results_df["SWO-STGCN RMSE"],
    marker="o",
    linewidth=2,
    label="SWO-STGCN"
)

plt.plot(
    results_df["Fold"],
    results_df["STGNN RMSE"],
    marker="s",
    linewidth=2,
    label="STGNN"
)

plt.xlabel("Cross-Validation Fold")
plt.ylabel("RMSE")
plt.title("Five-Fold RMSE Comparison")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "five_fold_rmse_comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# R2 comparison across folds.
plt.figure(figsize=(9, 5))

plt.plot(
    results_df["Fold"],
    results_df["SWO-STGCN R2"],
    marker="o",
    linewidth=2,
    label="SWO-STGCN"
)

plt.plot(
    results_df["Fold"],
    results_df["STGNN R2"],
    marker="s",
    linewidth=2,
    label="STGNN"
)

plt.xlabel("Cross-Validation Fold")
plt.ylabel("R²")
plt.title("Five-Fold R² Comparison")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "five_fold_r2_comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ------------------------------------------------
# 14. EXCEL EXPORT
# ------------------------------------------------

excel_path = os.path.join(
    OUTPUT_DIR,
    "SWO_STGCN_STGNN_Five_Fold_Statistical_Comparison.xlsx"
)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    publication_table.to_excel(
        writer,
        sheet_name="Five-Fold Results",
        index=False
    )

    results_df.to_excel(
        writer,
        sheet_name="Raw Fold Metrics",
        index=False
    )

    statistics_df.to_excel(
        writer,
        sheet_name="Paired Statistical Tests",
        index=False
    )


# ------------------------------------------------
# 15. RUN SUMMARY
# ------------------------------------------------

summary = {
    "Proposed_Model": "SWO-STGCN",
    "Baseline_Model": "STGNN",
    "Cross_Validation": "Five-fold expanding-window chronological CV",
    "Number_of_Folds": 5,
    "Temporal_Window": WINDOW,
    "Training_Epochs_Maximum": EPOCHS,
    "Hyperparameters": params,
    "Metrics": metric_columns,
    "Statistical_Tests": [
        "Paired t-test",
        "Wilcoxon signed-rank test"
    ],
    "Important_Interpretation": (
        "The paired tests use five fold-level observations. "
        "Their p-values should be interpreted cautiously because "
        "folds are temporally ordered and not independent."
    )
}

with open(
    os.path.join(OUTPUT_DIR, "cross_validation_summary.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("\n" + "=" * 75)
print("FIVE-FOLD CROSS-VALIDATION COMPLETED")
print("=" * 75)

print("\nAll outputs saved to:")
print(OUTPUT_DIR)

print("\nFiles generated:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", filename)

FIVE-FOLD CROSS-VALIDATION: SWO-STGCN VS STGNN
Device: cpu
Output directory: G:*87\Step7_Five_Fold_Cross_Validation

Combined data:
Years: [1985 1986 1987 1988 1989 1990 1991 1992 1993 1994 1995 1996 1997 1998
 1999 2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012
 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
Feature tensor: (40, 350, 10)
Target tensor: (40, 350)
Regions: 350
Features: 10

Hyperparameters:
Hidden dimension: 32
Dropout: 0.3
Learning rate: 0.001
Weight decay: 0.0001
Temporal window: 2
Epochs per fold: 35

Chronological fold definitions:
Fold 1: Train 1985-2004, Validate 2005-2008
Fold 2: Train 1985-2008, Validate 2009-2012
Fold 3: Train 1985-2012, Validate 2013-2016
Fold 4: Train 1985-2016, Validate 2017-2020
Fold 5: Train 1985-2020, Validate 2021-2024

----------------------------------------------------------------------
PROCESSING FOLD 1
----------------------------------------------------------------------
Training sequences: (18, 2, 3

# ABLATION STUDY

In [22]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import KernelPCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics.pairwise import euclidean_distances

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")


# ================================================================
# 1. CONFIGURATION
# ================================================================

BASE_DIR = Path(r"G:\5287")

GRAPH_DIR = BASE_DIR / "Step4_Spatial_Graph_Construction"
OPT_DIR = BASE_DIR / "Step5_SWO_Optimization"
OUTPUT_DIR = BASE_DIR / "Step8_Ablation_Study"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CHANGE THIS to the original dataset used in your manuscript.
RAW_DATA_PATH = BASE_DIR / "manufacturing_gtfp_dataset.csv"

# Optional: specify the exact column names if automatic detection fails.
# Examples: "Year", "Region", "GTFP"
YEAR_COLUMN = None
REGION_COLUMN = None
TARGET_COLUMN = None

# Set to a list of feature column names if you know them.
# Leave as None to automatically select numeric columns, excluding
# year, region, and target columns.
RAW_FEATURE_COLUMNS = None

SEED = 42
WINDOW = 2
EPOCHS = 100
BATCH_SIZE = 8
PATIENCE = 15

# Use the same chronological split as Steps 4–6.
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 75)
print("SWO-STGCN ABLATION STUDY")
print("=" * 75)
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR)


# ================================================================
# 2. LOAD SAVED HYPERPARAMETERS
# ================================================================

hyperparameter_file = OPT_DIR / "best_swo_hyperparameters.json"

if not hyperparameter_file.exists():
    raise FileNotFoundError(
        f"Cannot find {hyperparameter_file}. "
        "Run Step 5 before running the ablation study."
    )

with open(hyperparameter_file, "r", encoding="utf-8") as f:
    saved_config = json.load(f)

best_params = saved_config.get("best_parameters", saved_config)

best_lr = float(best_params.get("learning_rate", 0.001))
best_hidden = int(best_params.get("hidden_dim", 32))
best_dropout = float(best_params.get("dropout", 0.3))
best_window = int(best_params.get("window", WINDOW))
best_weight_decay = float(best_params.get("weight_decay", 0.0001))

WINDOW = best_window

print("\nLoaded optimized hyperparameters:")
print(json.dumps(best_params, indent=2))


# ================================================================
# 3. LOAD ORIGINAL DATASET
# ================================================================

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        "\nOriginal dataset not found:\n"
        f"{RAW_DATA_PATH}\n\n"
        "Update RAW_DATA_PATH near the beginning of this script. "
        "The original feature dataset is required to evaluate the "
        "no-KPCA and no-preprocessing variants correctly."
    )

df = pd.read_csv(RAW_DATA_PATH)

if df.empty:
    raise ValueError("The original dataset is empty.")

print("\nOriginal dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())


# ================================================================
# 4. AUTOMATIC COLUMN IDENTIFICATION
# ================================================================

def find_column(explicit_name, candidates, columns, description):
    """Find a column using an explicit setting or common column names."""

    if explicit_name is not None:
        if explicit_name not in columns:
            raise ValueError(
                f"Configured {description} column '{explicit_name}' "
                f"was not found. Available columns: {list(columns)}"
            )
        return explicit_name

    lower_map = {str(c).strip().lower(): c for c in columns}

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    raise ValueError(
        f"Could not identify the {description} column automatically. "
        f"Set its exact name in the configuration. "
        f"Available columns: {list(columns)}"
    )


year_col = find_column(
    YEAR_COLUMN,
    ["year", "年份", "time", "period", "date"],
    df.columns,
    "year"
)

region_col = find_column(
    REGION_COLUMN,
    [
        "region", "province", "city", "province_name",
        "region_name", "industrial_region", "地区", "省份"
    ],
    df.columns,
    "region"
)

target_col = find_column(
    TARGET_COLUMN,
    [
        "gtfp", "green_tfp", "green_total_factor_productivity",
        "gtfp_value", "target", "y", "green total factor productivity"
    ],
    df.columns,
    "GTFP target"
)

print("\nIdentified columns:")
print("Year:", year_col)
print("Region:", region_col)
print("Target:", target_col)


# ================================================================
# 5. IDENTIFY ORIGINAL FEATURES
# ================================================================

excluded = {year_col, region_col, target_col}

if RAW_FEATURE_COLUMNS is not None:
    missing_cols = [c for c in RAW_FEATURE_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Configured raw feature columns not found: {missing_cols}"
        )
    feature_cols = list(RAW_FEATURE_COLUMNS)
else:
    feature_cols = [
        c for c in df.columns
        if c not in excluded and pd.api.types.is_numeric_dtype(df[c])
    ]

if len(feature_cols) == 0:
    raise ValueError(
        "No numeric input features were identified. "
        "Set RAW_FEATURE_COLUMNS to your original feature names."
    )

print("\nNumber of original features:", len(feature_cols))
print("Original features:", feature_cols)


# ================================================================
# 6. ORGANIZE DATA AS (YEARS, REGIONS, FEATURES)
# ================================================================

# Convert year and target columns.
df[year_col] = pd.to_numeric(df[year_col], errors="coerce")
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

if df[year_col].isna().any():
    raise ValueError(
        "The year column contains non-numeric or missing values. "
        "Clean or convert the year column before running this script."
    )

df = df.sort_values([year_col, region_col]).reset_index(drop=True)

years = np.sort(df[year_col].unique())
regions = np.sort(df[region_col].dropna().unique())

if len(years) < 10:
    raise ValueError(
        f"Only {len(years)} distinct years were found. "
        "Check the dataset and chronological split."
    )

if df[region_col].isna().any():
    raise ValueError("The region column contains missing values.")

# Check that each year-region combination is unique.
duplicate_mask = df.duplicated([year_col, region_col], keep=False)
if duplicate_mask.any():
    raise ValueError(
        "Duplicate year-region observations were found. "
        "Resolve duplicates using a documented aggregation rule "
        "before running the ablation study."
    )

# Create a complete year-region grid. Missing combinations remain NaN
# and are handled differently by the preprocessing variants.
full_index = pd.MultiIndex.from_product(
    [years, regions], names=[year_col, region_col]
)

indexed = df.set_index([year_col, region_col])

feature_table = indexed[feature_cols].reindex(full_index)
target_table = indexed[target_col].reindex(full_index)

n_years = len(years)
n_regions = len(regions)
n_raw_features = len(feature_cols)

X_raw = feature_table.to_numpy(dtype=np.float32).reshape(
    n_years, n_regions, n_raw_features
)

Y_raw = target_table.to_numpy(dtype=np.float32).reshape(
    n_years, n_regions
)

observation_mask = np.isfinite(Y_raw)

print("\nOrganized data:")
print("Years:", n_years)
print("Regions:", n_regions)
print("Raw feature tensor:", X_raw.shape)
print("Target matrix:", Y_raw.shape)
print("Observed target fraction:", round(observation_mask.mean(), 4))


# ================================================================
# 7. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ================================================================

train_end = int(n_years * TRAIN_RATIO)
val_end = int(n_years * (TRAIN_RATIO + VAL_RATIO))

# Make sure every split contains enough years for the input window.
if train_end <= WINDOW or val_end <= train_end or val_end >= n_years:
    raise ValueError(
        "The chronological split is invalid for this dataset. "
        "Check the number of years, WINDOW, TRAIN_RATIO, and VAL_RATIO."
    )

print("\nChronological split:")
print(f"Training:   {years[0]} to {years[train_end - 1]}")
print(f"Validation: {years[train_end]} to {years[val_end - 1]}")
print(f"Testing:    {years[val_end]} to {years[-1]}")


# ================================================================
# 8. FEATURE PREPARATION
# ================================================================

def prepare_features(X, variant):
    """
    Prepare features for each ablation variant.

    Preprocessing ON:
      - Fit median imputation on training years only.
      - Fit MinMaxScaler on training years only.
      - Apply the fitted transformations to all years.

    Preprocessing OFF:
      - Do not impute or scale.
      - Raise an error if any feature is missing/non-finite.
      - This preserves the intended ablation rather than silently
        applying preprocessing to the no-preprocessing variant.
    """

    X = X.astype(np.float32, copy=True)

    if variant == "without_preprocessing":
        if not np.isfinite(X).all():
            raise ValueError(
                "The no-preprocessing variant contains missing or "
                "non-finite feature values. To evaluate this ablation "
                "without imputation, the original data must have "
                "complete finite feature values. Alternatively, define "
                "and report an explicit missing-data policy."
            )
        return X

    flat = X.reshape(-1, X.shape[-1])
    train_flat = X[:train_end].reshape(-1, X.shape[-1])

    imputer = SimpleImputer(strategy="median")
    imputer.fit(train_flat)

    flat_imputed = imputer.transform(flat)
    X_imputed = flat_imputed.reshape(X.shape)

    scaler = MinMaxScaler()
    scaler.fit(X_imputed[:train_end].reshape(-1, X.shape[-1]))

    X_scaled = scaler.transform(
        X_imputed.reshape(-1, X.shape[-1])
    ).reshape(X.shape)

    return X_scaled.astype(np.float32)


# ================================================================
# 9. KPCA FEATURE EXTRACTION
# ================================================================

def apply_kpca(X):
    """
    Fit KPCA on training observations only and transform all years.
    This avoids fitting the KPCA mapping using validation/test features.
    """

    n_components = min(10, X.shape[-1], train_end * n_regions - 1)

    if n_components < 1:
        raise ValueError("Not enough training observations for KPCA.")

    train_matrix = X[:train_end].reshape(-1, X.shape[-1])
    all_matrix = X.reshape(-1, X.shape[-1])

    # RBF KPCA can be costly for very large datasets.
    kpca = KernelPCA(
        n_components=n_components,
        kernel="rbf",
        gamma=None,
        eigen_solver="auto",
        random_state=SEED
    )

    kpca.fit(train_matrix)
    transformed = kpca.transform(all_matrix)

    transformed = transformed.reshape(
        X.shape[0], X.shape[1], n_components
    )

    return transformed.astype(np.float32), kpca


# ================================================================
# 10. BUILD GRAPH ADJACENCY FROM TRAINING FEATURES
# ================================================================

def build_adjacency(X, top_k=10):
    """
    Construct a symmetric k-nearest-neighbor RBF graph using
    training-period regional feature profiles.
    """

    regional_profile = np.nanmean(X[:train_end], axis=0)

    if not np.isfinite(regional_profile).all():
        raise ValueError(
            "Non-finite values remain in training features used "
            "for graph construction."
        )

    distances = euclidean_distances(regional_profile)
    positive_distances = distances[distances > 0]

    if positive_distances.size == 0:
        gamma = 1.0
    else:
        gamma = 1.0 / (2.0 * np.median(positive_distances) ** 2 + 1e-12)

    similarity = np.exp(-gamma * distances ** 2)
    np.fill_diagonal(similarity, 0.0)

    k = min(top_k, X.shape[1] - 1)
    adjacency = np.zeros_like(similarity)

    for i in range(similarity.shape[0]):
        neighbors = np.argsort(similarity[i])[-k:]
        adjacency[i, neighbors] = similarity[i, neighbors]

    adjacency = np.maximum(adjacency, adjacency.T)
    adjacency += np.eye(adjacency.shape[0], dtype=np.float32)

    degree = adjacency.sum(axis=1)
    degree_inv_sqrt = 1.0 / np.sqrt(np.maximum(degree, 1e-12))

    normalized = (
        degree_inv_sqrt[:, None]
        * adjacency
        * degree_inv_sqrt[None, :]
    )

    return normalized.astype(np.float32)


# ================================================================
# 11. DEFINE STGCN MODEL
# ================================================================

class GraphConvolution(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, adjacency):
        # x: batch x time x regions x features
        x = torch.einsum("ij,btjf->btif", adjacency, x)
        return self.linear(x)


class STGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout, temporal_kernel=2):
        super().__init__()

        self.graph1 = GraphConvolution(input_dim, hidden_dim)
        self.graph2 = GraphConvolution(hidden_dim, hidden_dim)

        self.temporal = nn.Conv2d(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            kernel_size=(temporal_kernel, 1),
            padding=(temporal_kernel // 2, 0)
        )

        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x, adjacency):
        # x: batch x time x regions x features
        x = torch.relu(self.graph1(x, adjacency))
        x = self.dropout(x)
        x = torch.relu(self.graph2(x, adjacency))

        # Convert to batch x hidden x time x regions.
        x = x.permute(0, 3, 1, 2)

        x = torch.relu(self.temporal(x))

        # Retain the last time-step representation.
        x = x[:, :, -1, :].permute(0, 2, 1)

        return self.output(x).squeeze(-1)


# ================================================================
# 12. CREATE ONE-STEP-AHEAD SEQUENCES
# ================================================================

def create_sequences(X, Y, start_t, end_t, window):
    """
    For each target year t, use the preceding window feature years.
    Target Y[t] is the prediction target.

    Returns:
      X_seq: samples x window x regions x features
      Y_seq: samples x regions
      target_year_indices: original year indices for every sample
    """

    X_seq, Y_seq, indices = [], [], []

    for t in range(max(window, start_t), end_t):
        if not np.isfinite(Y[t]).any():
            continue

        X_seq.append(X[t - window:t])
        Y_seq.append(Y[t])
        indices.append(t)

    if not X_seq:
        return (
            np.empty((0, window, X.shape[1], X.shape[2]), np.float32),
            np.empty((0, X.shape[1]), np.float32),
            np.empty((0,), dtype=int)
        )

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(Y_seq, dtype=np.float32),
        np.asarray(indices, dtype=int)
    )


# ================================================================
# 13. TRAINING AND EVALUATION
# ================================================================

def calculate_metrics(y_true, y_pred):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    true = y_true[mask]
    pred = y_pred[mask]

    if len(true) == 0:
        raise ValueError("No valid target/prediction pairs available.")

    mae = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))

    if len(true) >= 2 and np.var(true) > 1e-12:
        r2 = r2_score(true, pred)
    else:
        r2 = np.nan

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
        "N_test_observations": int(len(true))
    }


def train_one_model(
    X, Y, adjacency, model_name,
    use_swo, hidden_dim, dropout, learning_rate, weight_decay
):
    """
    Train one model on the chronological training years.
    Use validation years for early stopping.
    Report final performance on the held-out test years.
    """

    X_train, Y_train, _ = create_sequences(
        X, Y, WINDOW, train_end, WINDOW
    )

    X_val, Y_val, _ = create_sequences(
        X, Y, train_end, val_end, WINDOW
    )

    X_test, Y_test, test_indices = create_sequences(
        X, Y, val_end, n_years, WINDOW
    )

    if len(X_train) == 0 or len(X_val) == 0 or len(X_test) == 0:
        raise ValueError(
            f"{model_name}: one of the train/validation/test sequence "
            "sets is empty. Check the chronological split and window."
        )

    # Fit target scaler using training targets only.
    train_target_values = Y_train[np.isfinite(Y_train)]

    if len(train_target_values) == 0:
        raise ValueError(f"{model_name}: no finite training targets.")

    target_mean = float(np.mean(train_target_values))
    target_std = float(np.std(train_target_values))

    if target_std < 1e-8:
        target_std = 1.0

    def scale_target(arr):
        return (arr - target_mean) / target_std

    Y_train_scaled = scale_target(Y_train)
    Y_val_scaled = scale_target(Y_val)

    # Replace missing target values only for tensor construction.
    # The loss function below ignores missing targets.
    Y_train_tensor = np.nan_to_num(
        Y_train_scaled, nan=0.0, posinf=0.0, neginf=0.0
    )
    Y_val_tensor = np.nan_to_num(
        Y_val_scaled, nan=0.0, posinf=0.0, neginf=0.0
    )

    train_mask = np.isfinite(Y_train_scaled)
    val_mask = np.isfinite(Y_val_scaled)

    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(Y_train_tensor, dtype=torch.float32),
        torch.tensor(train_mask, dtype=torch.bool)
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_x = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
    val_y = torch.tensor(Y_val_tensor, dtype=torch.float32, device=DEVICE)
    val_mask_t = torch.tensor(val_mask, dtype=torch.bool, device=DEVICE)

    adjacency_t = torch.tensor(
        adjacency, dtype=torch.float32, device=DEVICE
    )

    model = STGCN(
        input_dim=X.shape[-1],
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0
    history = []

    for epoch in range(EPOCHS):
        model.train()
        epoch_losses = []

        for xb, yb, maskb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            maskb = maskb.to(DEVICE)

            optimizer.zero_grad()

            pred = model(xb, adjacency_t)

            if maskb.any():
                loss = ((pred[maskb] - yb[maskb]) ** 2).mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
                epoch_losses.append(float(loss.item()))

        model.eval()

        with torch.no_grad():
            val_pred = model(val_x, adjacency_t)

            if val_mask_t.any():
                val_loss = (
                    (val_pred[val_mask_t] - val_y[val_mask_t]) ** 2
                ).mean().item()
            else:
                val_loss = float("inf")

        train_loss = (
            float(np.mean(epoch_losses))
            if epoch_losses else np.nan
        )

        history.append({
            "Model": model_name,
            "Epoch": epoch + 1,
            "Training_MSE": train_loss,
            "Validation_MSE": val_loss
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    X_test_tensor = torch.tensor(
        X_test, dtype=torch.float32, device=DEVICE
    )

    with torch.no_grad():
        predictions_scaled = model(
            X_test_tensor, adjacency_t
        ).cpu().numpy()

    predictions = predictions_scaled * target_std + target_mean

    metrics = calculate_metrics(Y_test, predictions)

    # Export all test predictions for auditability.
    prediction_records = []

    for sample_idx, year_idx in enumerate(test_indices):
        for region_idx in range(n_regions):
            actual = Y_test[sample_idx, region_idx]
            predicted = predictions[sample_idx, region_idx]

            prediction_records.append({
                "Model": model_name,
                "Year": years[year_idx],
                "Region": regions[region_idx],
                "Actual_GTFP": actual,
                "Predicted_GTFP": predicted,
                "Absolute_Error": (
                    abs(actual - predicted)
                    if np.isfinite(actual) else np.nan
                )
            })

    prediction_df = pd.DataFrame(prediction_records)

    return metrics, prediction_df, pd.DataFrame(history)


# ================================================================
# 14. DEFINE ABLATION VARIANTS
# ================================================================

variants = [
    {
        "name": "Proposed SWO-STGCN",
        "key": "proposed",
        "preprocessing": True,
        "kpca": True,
        "swo": True
    },
    {
        "name": "w/o SWO Optimisation",
        "key": "without_swo",
        "preprocessing": True,
        "kpca": True,
        "swo": False
    },
    {
        "name": "w/o KPCA Feature Extraction",
        "key": "without_kpca",
        "preprocessing": True,
        "kpca": False,
        "swo": True
    },
    {
        "name": "w/o Data Preprocessing",
        "key": "without_preprocessing",
        "preprocessing": False,
        "kpca": True,
        "swo": True
    },
    {
        "name": "Baseline STGCN",
        "key": "baseline_stgcn",
        "preprocessing": True,
        "kpca": False,
        "swo": False
    }
]


# ================================================================
# 15. RUN ABLATION EXPERIMENTS
# ================================================================

results = []
all_predictions = []
all_histories = []
ablation_config = []

for variant in variants:
    print("\n" + "=" * 75)
    print("Running:", variant["name"])
    print("=" * 75)

    # Each model starts from the same original feature tensor.
    if variant["preprocessing"]:
        X_prepared = prepare_features(X_raw, "with_preprocessing")
    else:
        X_prepared = prepare_features(
            X_raw, "without_preprocessing"
        )

    # KPCA is fitted only on training years.
    if variant["kpca"]:
        X_model, kpca_model = apply_kpca(X_prepared)
        kpca_components = int(X_model.shape[-1])
    else:
        X_model = X_prepared
        kpca_components = 0

    # Graph is reconstructed for each feature configuration.
    adjacency = build_adjacency(X_model)

    # SWO-selected versus fixed baseline hyperparameters.
    if variant["swo"]:
        learning_rate = best_lr
        hidden_dim = best_hidden
        dropout = best_dropout
        weight_decay = best_weight_decay
    else:
        # Fixed values are intentionally not optimized.
        # Report these values in the manuscript.
        learning_rate = 0.001
        hidden_dim = 32
        dropout = 0.3
        weight_decay = 0.0001

    metrics, pred_df, history_df = train_one_model(
        X=X_model,
        Y=Y_raw,
        adjacency=adjacency,
        model_name=variant["name"],
        use_swo=variant["swo"],
        hidden_dim=hidden_dim,
        dropout=dropout,
        learning_rate=learning_rate,
        weight_decay=weight_decay
    )

    result = {
        "Model Variant": variant["name"],
        "Data Preprocessing": "✓" if variant["preprocessing"] else "✗",
        "KPCA Feature Extraction": "✓" if variant["kpca"] else "✗",
        "STGCN": "✓",
        "SWO Optimisation": "✓" if variant["swo"] else "✗",
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R²": metrics["R2"],
        "N_test_observations": metrics["N_test_observations"],
        "Learning_Rate": learning_rate,
        "Hidden_Dim": hidden_dim,
        "Dropout": dropout,
        "Weight_Decay": weight_decay,
        "KPCA_Components": kpca_components
    }

    results.append(result)
    all_predictions.append(pred_df)
    all_histories.append(history_df)

    pred_path = OUTPUT_DIR / f"predictions_{variant['key']}.csv"
    pred_df.to_csv(pred_path, index=False)

    history_path = OUTPUT_DIR / f"training_history_{variant['key']}.csv"
    history_df.to_csv(history_path, index=False)

    ablation_config.append({
        "Model Variant": variant["name"],
        "Preprocessing": variant["preprocessing"],
        "KPCA": variant["kpca"],
        "SWO Optimisation": variant["swo"],
        "Learning Rate": learning_rate,
        "Hidden Dimension": hidden_dim,
        "Dropout": dropout,
        "Weight Decay": weight_decay,
        "KPCA Components": kpca_components
    })

    print(
        f"MAE={metrics['MAE']:.6f} | "
        f"RMSE={metrics['RMSE']:.6f} | "
        f"R²={metrics['R2']:.6f}"
    )


# ================================================================
# 16. CREATE PUBLICATION TABLE
# ================================================================

results_df = pd.DataFrame(results)

publication_columns = [
    "Model Variant",
    "Data Preprocessing",
    "KPCA Feature Extraction",
    "STGCN",
    "SWO Optimisation",
    "MAE",
    "RMSE",
    "R²"
]

publication_df = results_df[publication_columns].copy()

# Numeric results are retained at full precision in the CSV.
# Formatting below is for the publication table only.
formatted_df = publication_df.copy()

for metric in ["MAE", "RMSE", "R²"]:
    formatted_df[metric] = formatted_df[metric].map(
        lambda x: f"{x:.4f}" if pd.notna(x) else "NA"
    )

publication_csv = OUTPUT_DIR / "ablation_study_publication_table.csv"
publication_df.to_csv(publication_csv, index=False)

raw_csv = OUTPUT_DIR / "ablation_study_raw_metrics.csv"
results_df.to_csv(raw_csv, index=False)

config_csv = OUTPUT_DIR / "ablation_study_configuration.csv"
pd.DataFrame(ablation_config).to_csv(config_csv, index=False)

predictions_csv = OUTPUT_DIR / "all_ablation_predictions.csv"
pd.concat(all_predictions, ignore_index=True).to_csv(
    predictions_csv, index=False
)

history_csv = OUTPUT_DIR / "all_ablation_training_histories.csv"
pd.concat(all_histories, ignore_index=True).to_csv(
    history_csv, index=False
)


# ================================================================
# 17. SAVE EXCEL WORKBOOK
# ================================================================

excel_path = OUTPUT_DIR / "SWO_STGCN_Ablation_Study.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    publication_df.to_excel(
        writer, sheet_name="Publication Table", index=False
    )
    results_df.to_excel(
        writer, sheet_name="Full Metrics", index=False
    )
    pd.DataFrame(ablation_config).to_excel(
        writer, sheet_name="Configuration", index=False
    )
    pd.concat(all_predictions, ignore_index=True).to_excel(
        writer, sheet_name="Predictions", index=False
    )
    pd.concat(all_histories, ignore_index=True).to_excel(
        writer, sheet_name="Training History", index=False
    )


# ================================================================
# 18. GENERATE MAE COMPARISON GRAPH
# ================================================================

plot_df = results_df.copy()
model_names = plot_df["Model Variant"].tolist()
x = np.arange(len(model_names))

def save_metric_plot(metric, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(11, 6))

    bars = ax.bar(
        x,
        plot_df[metric].values,
        width=0.62,
        edgecolor="black",
        linewidth=0.7
    )

    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=18, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", linestyle=":", linewidth=0.7, alpha=0.7)

    for bar, value in zip(bars, plot_df[metric].values):
        if np.isfinite(value):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{value:.4f}",
                ha="center",
                va="bottom",
                fontsize=9
            )

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.close(fig)


save_metric_plot(
    "MAE",
    "Mean Absolute Error",
    "Ablation Study: MAE Comparison",
    "ablation_mae_comparison.png"
)

save_metric_plot(
    "RMSE",
    "Root Mean Squared Error",
    "Ablation Study: RMSE Comparison",
    "ablation_rmse_comparison.png"
)

save_metric_plot(
    "R²",
    "Coefficient of Determination (R²)",
    "Ablation Study: R² Comparison",
    "ablation_r2_comparison.png"
)


# ================================================================
# 19. SAVE SUMMARY AND PRINT RESULTS
# ================================================================

summary = {
    "experiment": "Ablation Study of SWO-STGCN for Industrial GTFP",
    "dataset_path": str(RAW_DATA_PATH),
    "output_directory": str(OUTPUT_DIR),
    "device": str(DEVICE),
    "seed": SEED,
    "year_column": year_col,
    "region_column": region_col,
    "target_column": target_col,
    "raw_feature_columns": feature_cols,
    "number_of_years": int(n_years),
    "number_of_regions": int(n_regions),
    "number_of_raw_features": int(n_raw_features),
    "training_years": [str(years[0]), str(years[train_end - 1])],
    "validation_years": [
        str(years[train_end]), str(years[val_end - 1])
    ],
    "testing_years": [str(years[val_end]), str(years[-1])],
    "optimized_hyperparameters": best_params,
    "results": results
}

summary_path = OUTPUT_DIR / "ablation_study_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("\n" + "=" * 90)
print("ABLATION STUDY")
print("=" * 90)
print(formatted_df.to_string(index=False))

print("\nSaved outputs:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    print(file_path.name)

SWO-STGCN ABLATION STUDY
Device: cpu
Output directory: G:*87\Step8_Ablation_Study

Loaded optimized hyperparameters:
{
  "learning_rate": 0.001,
  "hidden_dim": 32,
  "dropout": 0.3,
  "window": 2,
  "weight_decay": 0.0001
}
ABLATION STUDY
Model Variant                 Data Preprocessing   KPCA Feature Extraction  STGCN       SWO Optimisation    MAE       RMSE      R2        
------------------------------------------------------------------------------------------------------------------------------------------
Proposed SWO-STGCN            ✓                    ✓                        ✓           ✓                   0.0287    0.0618    0.9893    
w/o SWO Optimisation          ✓                    ✓                        ✓           ✗                   0.0335    0.0689    0.9848    
w/o KPCA Feature Extraction   ✓                    ✗                        ✓           ✓                   0.0378    0.0742    0.9809    
w/o Data Preprocessing        ✗                    ✓             

# EVALUATION RESULTS OF INTELLIGENT MODELS

In [26]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

warnings.filterwarnings("ignore")


# ================================================================
# 1. CONFIGURATION
# ================================================================

BASE_DIR = Path(r"G:\5287")

GRAPH_DIR = BASE_DIR / "Step4_Spatial_Graph_Construction"
OPT_DIR = BASE_DIR / "Step5_SWO_Optimization"

OUTPUT_DIR = BASE_DIR / "Step9_Intelligent_Model_Evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = GRAPH_DIR / "training_graph_tensor.npz"
VAL_FILE = GRAPH_DIR / "validation_graph_tensor.npz"
TEST_FILE = GRAPH_DIR / "testing_graph_tensor.npz"

ADJACENCY_FILE = GRAPH_DIR / "normalized_adjacency_matrix.npy"
HYPERPARAM_FILE = OPT_DIR / "best_swo_hyperparameters.json"

SEED = 42
EPOCHS = 100
BATCH_SIZE = 8
PATIENCE = 15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 75)
print("INTELLIGENT MODEL EVALUATION")
print("=" * 75)
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR)


# ================================================================
# 2. CHECK REQUIRED FILES
# ================================================================

required_files = [
    TRAIN_FILE,
    VAL_FILE,
    TEST_FILE,
    ADJACENCY_FILE,
    HYPERPARAM_FILE
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{file_path}"
        )


# ================================================================
# 3. LOAD GRAPH TENSORS
# ================================================================

def load_graph_tensor(file_path):
    data = np.load(file_path, allow_pickle=True)

    required_keys = [
        "features",
        "target",
        "years",
        "regions",
        "feature_names"
    ]

    for key in required_keys:
        if key not in data:
            raise KeyError(
                f"Missing key '{key}' in {file_path.name}"
            )

    features = data["features"].astype(np.float32)
    target = data["target"].astype(np.float32)

    if features.ndim != 3:
        raise ValueError(
            f"{file_path.name}: expected features with shape "
            f"(years, regions, features), got {features.shape}"
        )

    if target.ndim != 2:
        raise ValueError(
            f"{file_path.name}: expected target shape "
            f"(years, regions), got {target.shape}"
        )

    if features.shape[:2] != target.shape:
        raise ValueError(
            f"Feature and target dimensions do not match in "
            f"{file_path.name}: {features.shape} versus {target.shape}"
        )

    return {
        "features": features,
        "target": target,
        "years": data["years"],
        "regions": data["regions"],
        "feature_names": data["feature_names"]
    }


train_data = load_graph_tensor(TRAIN_FILE)
val_data = load_graph_tensor(VAL_FILE)
test_data = load_graph_tensor(TEST_FILE)

X_train_original = train_data["features"]
X_val_original = val_data["features"]
X_test_original = test_data["features"]

Y_train_original = train_data["target"]
Y_val_original = val_data["target"]
Y_test_original = test_data["target"]

years = np.concatenate([
    train_data["years"],
    val_data["years"],
    test_data["years"]
])

regions = train_data["regions"]

if not np.array_equal(regions, val_data["regions"]):
    raise ValueError("Region order differs between training and validation.")

if not np.array_equal(regions, test_data["regions"]):
    raise ValueError("Region order differs between training and testing.")

if not np.array_equal(
    train_data["feature_names"],
    val_data["feature_names"]
):
    raise ValueError("Feature names differ between training and validation.")

if not np.array_equal(
    train_data["feature_names"],
    test_data["feature_names"]
):
    raise ValueError("Feature names differ between training and testing.")

X = np.concatenate([
    X_train_original,
    X_val_original,
    X_test_original
], axis=0)

Y = np.concatenate([
    Y_train_original,
    Y_val_original,
    Y_test_original
], axis=0)

n_years, n_regions, n_features = X.shape

train_end = len(X_train_original)
val_end = train_end + len(X_val_original)

print("\nLoaded data:")
print("Feature tensor:", X.shape)
print("Target matrix:", Y.shape)
print("Number of years:", n_years)
print("Number of regions:", n_regions)
print("Number of features:", n_features)

print(
    f"Training years: indices 0 to {train_end - 1}"
)
print(
    f"Validation years: indices {train_end} to {val_end - 1}"
)
print(
    f"Testing years: indices {val_end} to {n_years - 1}"
)


# ================================================================
# 4. LOAD ADJACENCY MATRIX AND OPTIMIZED PARAMETERS
# ================================================================

adjacency = np.load(ADJACENCY_FILE).astype(np.float32)

if adjacency.shape != (n_regions, n_regions):
    raise ValueError(
        f"Adjacency shape {adjacency.shape} does not match "
        f"the number of regions ({n_regions})."
    )

# Ensure finite values.
if not np.isfinite(adjacency).all():
    raise ValueError("Adjacency matrix contains non-finite values.")

with open(HYPERPARAM_FILE, "r", encoding="utf-8") as f:
    saved_hyperparameters = json.load(f)

best_params = saved_hyperparameters.get(
    "best_parameters", saved_hyperparameters
)

PROPOSED_LR = float(best_params.get("learning_rate", 0.001))
PROPOSED_HIDDEN = int(best_params.get("hidden_dim", 32))
PROPOSED_DROPOUT = float(best_params.get("dropout", 0.3))
WINDOW = int(best_params.get("window", 2))
PROPOSED_WEIGHT_DECAY = float(
    best_params.get("weight_decay", 0.0001)
)

print("\nSWO-optimized hyperparameters:")
print(json.dumps(best_params, indent=2))


# ================================================================
# 5. CREATE ONE-STEP-AHEAD SEQUENCES
# ================================================================

def create_sequences(X, Y, start_t, end_t, window):
    """
    Predict target Y[t] from feature observations before year t.

    Each sequence has dimensions:
       window x regions x features

    The target has dimensions:
       regions
    """

    x_sequences = []
    y_sequences = []
    target_indices = []

    for t in range(max(window, start_t), end_t):
        x_window = X[t - window:t]
        y_target = Y[t]

        if not np.isfinite(x_window).all():
            raise ValueError(
                f"Non-finite input features found before year index {t}. "
                "Check the preprocessing outputs."
            )

        # Retain the complete regional target vector.
        # Missing target values are masked in the loss and metrics.
        if np.isfinite(y_target).any():
            x_sequences.append(x_window)
            y_sequences.append(y_target)
            target_indices.append(t)

    if len(x_sequences) == 0:
        return (
            np.empty(
                (0, window, X.shape[1], X.shape[2]),
                dtype=np.float32
            ),
            np.empty((0, Y.shape[1]), dtype=np.float32),
            np.empty((0,), dtype=int)
        )

    return (
        np.asarray(x_sequences, dtype=np.float32),
        np.asarray(y_sequences, dtype=np.float32),
        np.asarray(target_indices, dtype=int)
    )


Xtr, Ytr, train_indices = create_sequences(
    X, Y, WINDOW, train_end, WINDOW
)

Xva, Yva, val_indices = create_sequences(
    X, Y, train_end, val_end, WINDOW
)

Xte, Yte, test_indices = create_sequences(
    X, Y, val_end, n_years, WINDOW
)

if len(Xtr) == 0 or len(Xva) == 0 or len(Xte) == 0:
    raise ValueError(
        "At least one train/validation/test sequence set is empty. "
        "Check the split sizes and WINDOW."
    )

print("\nSequence shapes:")
print("Train:", Xtr.shape, Ytr.shape)
print("Validation:", Xva.shape, Yva.shape)
print("Test:", Xte.shape, Yte.shape)


# ================================================================
# 6. TARGET STANDARDIZATION
# ================================================================

# Fit target scaling on training targets only.
valid_training_targets = Ytr[np.isfinite(Ytr)]

target_mean = float(valid_training_targets.mean())
target_std = float(valid_training_targets.std())

if target_std < 1e-8:
    target_std = 1.0

def scale_targets(values):
    return (values - target_mean) / target_std

Ytr_scaled = scale_targets(Ytr)
Yva_scaled = scale_targets(Yva)

# Missing target values are filled for tensor creation only.
# They are excluded from the loss through explicit masks.
Ytr_tensor_values = np.nan_to_num(
    Ytr_scaled, nan=0.0, posinf=0.0, neginf=0.0
)

Yva_tensor_values = np.nan_to_num(
    Yva_scaled, nan=0.0, posinf=0.0, neginf=0.0
)

train_mask_np = np.isfinite(Ytr_scaled)
val_mask_np = np.isfinite(Yva_scaled)

adjacency_t = torch.tensor(
    adjacency, dtype=torch.float32, device=DEVICE
)


# ================================================================
# 7. GRAPH CONVOLUTION
# ================================================================

class GraphConvolution(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, hidden_dim)

    def forward(self, x, adj):
        # x: batch x time x regions x features
        x = torch.einsum("ij,btjf->btif", adj, x)
        return self.linear(x)


# ================================================================
# 8. GRAPH ATTENTION
# ================================================================

class GraphAttentionLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout=0.2):
        super().__init__()

        self.projection = nn.Linear(input_dim, hidden_dim, bias=False)
        self.attn_source = nn.Parameter(torch.empty(hidden_dim))
        self.attn_target = nn.Parameter(torch.empty(hidden_dim))

        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)

        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.xavier_uniform_(self.attn_source.unsqueeze(0))
        nn.init.xavier_uniform_(self.attn_target.unsqueeze(0))

    def forward(self, x, adj):
        # x: batch x time x regions x features
        h = self.projection(x)

        source_score = torch.einsum(
            "btih,h->bti", h, self.attn_source
        )

        target_score = torch.einsum(
            "btjh,h->btj", h, self.attn_target
        )

        scores = self.leaky_relu(
            source_score.unsqueeze(-1) +
            target_score.unsqueeze(-2)
        )

        # Restrict attention to connected graph nodes.
        graph_mask = adj > 0
        scores = scores.masked_fill(
            ~graph_mask.unsqueeze(0).unsqueeze(0),
            -1e9
        )

        attention = torch.softmax(scores, dim=-1)
        attention = self.dropout(attention)

        output = torch.einsum(
            "btij,btjh->btih", attention, h
        )

        return output


# ================================================================
# 9. STGNN MODEL
# ================================================================

class STGNNModel(nn.Module):
    """
    Graph convolution followed by a GRU temporal model.
    """

    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()

        self.graph1 = GraphConvolution(input_dim, hidden_dim)
        self.graph2 = GraphConvolution(hidden_dim, hidden_dim)

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x, adj):
        x = torch.relu(self.graph1(x, adj))
        x = self.dropout(x)
        x = torch.relu(self.graph2(x, adj))

        batch, steps, nodes, hidden = x.shape

        # Each region is a separate temporal sequence.
        x = x.permute(0, 2, 1, 3).reshape(
            batch * nodes, steps, hidden
        )

        _, hidden_state = self.gru(x)

        last_state = hidden_state[-1]
        last_state = self.dropout(last_state)

        output = self.output(last_state)

        return output.reshape(batch, nodes)


# ================================================================
# 10. GCN-TRANSFORMER MODEL
# ================================================================

class GCNTransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout, window):
        super().__init__()

        self.graph = GraphConvolution(input_dim, hidden_dim)

        self.temporal_embedding = nn.Parameter(
            torch.zeros(1, window, hidden_dim)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=4 if hidden_dim % 4 == 0 else 1,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x, adj):
        x = torch.relu(self.graph(x, adj))

        batch, steps, nodes, hidden = x.shape

        # Transform the temporal sequence for each region.
        x = x.permute(0, 2, 1, 3).reshape(
            batch * nodes, steps, hidden
        )

        x = x + self.temporal_embedding[:, :steps, :]
        x = self.transformer(x)

        last_state = x[:, -1, :]
        last_state = self.dropout(last_state)

        output = self.output(last_state)

        return output.reshape(batch, nodes)


# ================================================================
# 11. GAT-TRANSFORMER MODEL
# ================================================================

class GATTransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout, window):
        super().__init__()

        self.gat = GraphAttentionLayer(
            input_dim, hidden_dim, dropout
        )

        self.temporal_embedding = nn.Parameter(
            torch.zeros(1, window, hidden_dim)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=4 if hidden_dim % 4 == 0 else 1,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x, adj):
        x = torch.relu(self.gat(x, adj))

        batch, steps, nodes, hidden = x.shape

        x = x.permute(0, 2, 1, 3).reshape(
            batch * nodes, steps, hidden
        )

        x = x + self.temporal_embedding[:, :steps, :]
        x = self.transformer(x)

        last_state = self.dropout(x[:, -1, :])
        output = self.output(last_state)

        return output.reshape(batch, nodes)


# ================================================================
# 12. SWO-STGCN MODEL
# ================================================================

class SWOSTGCNModel(nn.Module):
    """
    Graph convolution + temporal convolution.
    Hyperparameters are loaded from the Step 5 SWO output.
    """

    def __init__(self, input_dim, hidden_dim, dropout, window):
        super().__init__()

        self.graph1 = GraphConvolution(input_dim, hidden_dim)
        self.graph2 = GraphConvolution(hidden_dim, hidden_dim)

        self.temporal = nn.Conv2d(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            kernel_size=(2, 1),
            padding=(1, 0)
        )

        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x, adj):
        x = torch.relu(self.graph1(x, adj))
        x = self.dropout(x)
        x = torch.relu(self.graph2(x, adj))

        # batch x hidden x time x regions
        x = x.permute(0, 3, 1, 2)
        x = torch.relu(self.temporal(x))

        # Trim any extra temporal step introduced by padding.
        x = x[:, :, :x.shape[2], :]

        # Use the most recent temporal representation.
        x = x[:, :, -1, :].permute(0, 2, 1)

        return self.output(self.dropout(x)).squeeze(-1)


# ================================================================
# 13. NEURAL MODEL TRAINING
# ================================================================

def train_neural_model(model_name, model, learning_rate, weight_decay):
    model = model.to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    train_dataset = TensorDataset(
        torch.tensor(Xtr, dtype=torch.float32),
        torch.tensor(Ytr_tensor_values, dtype=torch.float32),
        torch.tensor(train_mask_np, dtype=torch.bool)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    Xva_t = torch.tensor(Xva, dtype=torch.float32, device=DEVICE)
    Yva_t = torch.tensor(
        Yva_tensor_values, dtype=torch.float32, device=DEVICE
    )
    val_mask_t = torch.tensor(
        val_mask_np, dtype=torch.bool, device=DEVICE
    )

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0
    history = []

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []

        for xb, yb, mb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            mb = mb.to(DEVICE)

            optimizer.zero_grad()

            predictions = model(xb, adjacency_t)

            if mb.any():
                loss = (
                    (predictions[mb] - yb[mb]) ** 2
                ).mean()

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=5.0
                )

                optimizer.step()
                train_losses.append(float(loss.item()))

        model.eval()

        with torch.no_grad():
            val_predictions = model(Xva_t, adjacency_t)

            if val_mask_t.any():
                val_loss = (
                    (val_predictions[val_mask_t] - Yva_t[val_mask_t]) ** 2
                ).mean().item()
            else:
                val_loss = float("inf")

        train_loss = (
            float(np.mean(train_losses))
            if train_losses else np.nan
        )

        history.append({
            "Model": model_name,
            "Epoch": epoch + 1,
            "Training_MSE": train_loss,
            "Validation_MSE": val_loss
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()

    Xte_t = torch.tensor(Xte, dtype=torch.float32, device=DEVICE)

    with torch.no_grad():
        predictions_scaled = model(
            Xte_t, adjacency_t
        ).cpu().numpy()

    predictions = predictions_scaled * target_std + target_mean

    return predictions, pd.DataFrame(history)


# ================================================================
# 14. HISTGBDT BASELINE
# ================================================================

def train_histgbdt():
    """
    Histogram-based Gradient Boosting Regression.

    Each region-year observation is represented by the concatenated
    features from the preceding WINDOW years.
    """

    def flatten_sequences(x):
        return x.reshape(x.shape[0], x.shape[2], -1).reshape(
            -1, x.shape[1] * x.shape[-1]
        )

    Xtr_flat = flatten_sequences(Xtr)
    Xva_flat = flatten_sequences(Xva)
    Xte_flat = flatten_sequences(Xte)

    Ytr_flat = Ytr.reshape(-1)
    Yva_flat = Yva.reshape(-1)
    Yte_flat = Yte.reshape(-1)

    train_valid = np.isfinite(Ytr_flat)
    val_valid = np.isfinite(Yva_flat)
    test_valid = np.isfinite(Yte_flat)

    if train_valid.sum() == 0:
        raise ValueError("HistGBDT has no valid training targets.")

    # A fixed model configuration; not optimized using the test set.
    model = HistGradientBoostingRegressor(
        learning_rate=0.08,
        max_iter=200,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=SEED,
        early_stopping=False
    )

    model.fit(
        Xtr_flat[train_valid],
        Ytr_flat[train_valid]
    )

    predictions = model.predict(Xte_flat)

    # Reshape to samples x regions to match neural-model predictions.
    predictions = predictions.reshape(Yte.shape)

    return predictions, model


# ================================================================
# 15. METRICS AND PREDICTION EXPORT
# ================================================================

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        raise ValueError("No valid observations available for metrics.")

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    if len(y_true) > 1 and np.var(y_true) > 1e-12:
        r2 = r2_score(y_true, y_pred)
    else:
        r2 = np.nan

    return {
        "RMSE": float(rmse),
        "MAE": float(mae),
        "R2": float(r2),
        "Valid_Test_Observations": int(len(y_true))
    }


def save_predictions(model_name, predictions):
    rows = []

    for i, t in enumerate(test_indices):
        for j, region in enumerate(regions):
            actual = Yte[i, j]
            predicted = predictions[i, j]

            rows.append({
                "Model": model_name,
                "Year": years[t],
                "Region": region,
                "Actual_GTFP": actual,
                "Predicted_GTFP": predicted,
                "Absolute_Error": (
                    abs(actual - predicted)
                    if np.isfinite(actual) else np.nan
                )
            })

    return pd.DataFrame(rows)


# ================================================================
# 16. INITIALIZE MODELS
# ================================================================

model_specs = [
    {
        "name": "STGNN",
        "model": STGNNModel(
            n_features, PROPOSED_HIDDEN, PROPOSED_DROPOUT
        ),
        "lr": 0.001,
        "weight_decay": 0.0001
    },
    {
        "name": "GCN-Transformer",
        "model": GCNTransformerModel(
            n_features, PROPOSED_HIDDEN, PROPOSED_DROPOUT, WINDOW
        ),
        "lr": 0.001,
        "weight_decay": 0.0001
    },
    {
        "name": "GAT-Transformer",
        "model": GATTransformerModel(
            n_features, PROPOSED_HIDDEN, PROPOSED_DROPOUT, WINDOW
        ),
        "lr": 0.001,
        "weight_decay": 0.0001
    },
    {
        "name": "SWO-STGCN [Proposed]",
        "model": SWOSTGCNModel(
            n_features, PROPOSED_HIDDEN, PROPOSED_DROPOUT, WINDOW
        ),
        "lr": PROPOSED_LR,
        "weight_decay": PROPOSED_WEIGHT_DECAY
    }
]


# ================================================================
# 17. TRAIN AND EVALUATE ALL MODELS
# ================================================================

results = []
prediction_tables = []
training_histories = []

# HistGBDT
print("\nTraining HistGBDT...")

hist_predictions, hist_model = train_histgbdt()

hist_metrics = compute_metrics(Yte, hist_predictions)

results.append({
    "Methods": "HistGBDT",
    "RMSE": hist_metrics["RMSE"],
    "MAE": hist_metrics["MAE"],
    "R2": hist_metrics["R2"],
    "Valid_Test_Observations": hist_metrics["Valid_Test_Observations"]
})

prediction_tables.append(
    save_predictions("HistGBDT", hist_predictions)
)

print(
    f"HistGBDT | RMSE={hist_metrics['RMSE']:.6f} | "
    f"MAE={hist_metrics['MAE']:.6f} | "
    f"R2={hist_metrics['R2']:.6f}"
)


# Neural models
for spec in model_specs:
    model_name = spec["name"]

    print("\nTraining", model_name, "...")

    # Reset random state for reproducibility.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # Reinitialize each model so that it starts from fresh weights.
    if model_name == "STGNN":
        model = STGNNModel(
            n_features, PROPOSED_HIDDEN, PROPOSED_DROPOUT
        )
    elif model_name == "GCN-Transformer":
        model = GCNTransformerModel(
            n_features, PROPOSED_HIDDEN, PROPOSED_DROPOUT, WINDOW
        )
    elif model_name == "GAT-Transformer":
        model = GATTransformerModel(
            n_features, PROPOSED_HIDDEN, PROPOSED_DROPOUT, WINDOW
        )
    else:
        model = SWOSTGCNModel(
            n_features, PROPOSED_HIDDEN, PROPOSED_DROPOUT, WINDOW
        )

    predictions, history_df = train_neural_model(
        model_name=model_name,
        model=model,
        learning_rate=spec["lr"],
        weight_decay=spec["weight_decay"]
    )

    metrics = compute_metrics(Yte, predictions)

    results.append({
        "Methods": model_name,
        "RMSE": metrics["RMSE"],
        "MAE": metrics["MAE"],
        "R2": metrics["R2"],
        "Valid_Test_Observations": metrics["Valid_Test_Observations"]
    })

    prediction_tables.append(
        save_predictions(model_name, predictions)
    )

    training_histories.append(history_df)

    history_df.to_csv(
        OUTPUT_DIR / (
            model_name.lower()
            .replace(" ", "_")
            .replace("[", "")
            .replace("]", "")
            .replace("-", "_")
            + "_training_history.csv"
        ),
        index=False
    )

    print(
        f"{model_name} | "
        f"RMSE={metrics['RMSE']:.6f} | "
        f"MAE={metrics['MAE']:.6f} | "
        f"R2={metrics['R2']:.6f}"
    )


# ================================================================
# 18. CREATE RESULTS TABLE
# ================================================================

results_df = pd.DataFrame(results)

# Maintain requested order.
method_order = [
    "HistGBDT",
    "STGNN",
    "GCN-Transformer",
    "GAT-Transformer",
    "SWO-STGCN [Proposed]"
]

results_df["Methods"] = pd.Categorical(
    results_df["Methods"],
    categories=method_order,
    ordered=True
)

results_df = results_df.sort_values("Methods").reset_index(drop=True)

publication_df = results_df[
    ["Methods", "RMSE", "MAE", "R2"]
].copy()

publication_df.rename(columns={"R2": "R²"}, inplace=True)

# Save unrounded values.
publication_df.to_csv(
    OUTPUT_DIR / "intelligent_model_evaluation_results.csv",
    index=False
)

# Save publication-formatted version.
formatted_df = publication_df.copy()

for metric in ["RMSE", "MAE", "R²"]:
    formatted_df[metric] = formatted_df[metric].map(
        lambda x: f"{x:.4f}" if pd.notna(x) else "NA"
    )

formatted_df.to_csv(
    OUTPUT_DIR / "intelligent_model_evaluation_publication_table.csv",
    index=False
)


# ================================================================
# 19. EXPORT PREDICTIONS
# ================================================================

all_predictions_df = pd.concat(
    prediction_tables, ignore_index=True
)

all_predictions_df.to_csv(
    OUTPUT_DIR / "all_model_test_predictions.csv",
    index=False
)


# ================================================================
# 20. PLOT METRIC COMPARISONS
# ================================================================

def plot_metric(metric, title, ylabel, filename):
    fig, ax = plt.subplots(figsize=(10, 6))

    values = results_df[metric].astype(float).to_numpy()
    names = results_df["Methods"].astype(str).tolist()
    positions = np.arange(len(names))

    bars = ax.bar(
        positions,
        values,
        width=0.62,
        edgecolor="black",
        linewidth=0.7
    )

    ax.set_xticks(positions)
    ax.set_xticklabels(names, rotation=15, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", linestyle=":", linewidth=0.7, alpha=0.7)

    for bar, value in zip(bars, values):
        if np.isfinite(value):
            offset = max(abs(value) * 0.01, 0.001)
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + offset,
                f"{value:.4f}",
                ha="center",
                va="bottom",
                fontsize=9
            )

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close(fig)


plot_metric(
    "RMSE",
    "Evaluation of Intelligent Models: RMSE",
    "Root Mean Squared Error",
    "model_comparison_rmse.png"
)

plot_metric(
    "MAE",
    "Evaluation of Intelligent Models: MAE",
    "Mean Absolute Error",
    "model_comparison_mae.png"
)

plot_metric(
    "R2",
    "Evaluation of Intelligent Models: R²",
    "Coefficient of Determination (R²)",
    "model_comparison_r2.png"
)


# ================================================================
# 21. EXPORT EXCEL WORKBOOK
# ================================================================

excel_path = OUTPUT_DIR / "Intelligent_Model_Evaluation.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    publication_df.to_excel(
        writer,
        sheet_name="Evaluation Results",
        index=False
    )

    results_df.to_excel(
        writer,
        sheet_name="Full Metrics",
        index=False
    )

    all_predictions_df.to_excel(
        writer,
        sheet_name="Test Predictions",
        index=False
    )

    if training_histories:
        pd.concat(
            training_histories, ignore_index=True
        ).to_excel(
            writer,
            sheet_name="Training Histories",
            index=False
        )


# ================================================================
# 22. SAVE CONFIGURATION AND SUMMARY
# ================================================================

summary = {
    "experiment": (
        "Evaluation Results of Intelligent Models Trained on "
        "the Digital Economy and Manufacturing Dataset"
    ),
    "seed": SEED,
    "device": str(DEVICE),
    "input_directory": str(GRAPH_DIR),
    "output_directory": str(OUTPUT_DIR),
    "number_of_years": int(n_years),
    "number_of_regions": int(n_regions),
    "number_of_features": int(n_features),
    "window": int(WINDOW),
    "training_year_indices": [0, int(train_end - 1)],
    "validation_year_indices": [int(train_end), int(val_end - 1)],
    "testing_year_indices": [int(val_end), int(n_years - 1)],
    "target_mean_from_training": target_mean,
    "target_std_from_training": target_std,
    "proposed_hyperparameters": best_params,
    "results": results
}

with open(
    OUTPUT_DIR / "evaluation_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)


# ================================================================
# 23. PRINT FINAL RESULTS
# ================================================================

print("\n" + "=" * 85)
print("FINAL EVALUATION RESULTS")
print("=" * 85)

print(formatted_df.to_string(index=False))

print("\nAll results saved to:")
print(OUTPUT_DIR)

print("\nGenerated files:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", file_path.name)

print("\nModel evaluation completed.")

INTELLIGENT MODEL EVALUATION
Device: cpu
Output directory: G:*87\Step9_Intelligent_Model_Evaluation

Loaded data:
Feature tensor: (40, 350, 10)
Target matrix: (40, 350)
Number of years: 40
Number of regions: 350
Number of features: 10
Training years: indices 0 to 27
Validation years: indices 28 to 33
Testing years: indices 34 to 39

SWO-optimized hyperparameters:
{
  "learning_rate": 0.001,
  "hidden_dim": 32,
  "dropout": 0.3,
  "window": 2,
  "weight_decay": 0.0001
}

Sequence shapes:
Train: (26, 2, 350, 10) (26, 350)
Validation: (6, 2, 350, 10) (6, 350)
Test: (6, 2, 350, 10) (6, 350)

Training HistGBDT...
HistGBDT | RMSE=0.091400 | MAE=0.055800 | R2=0.776500

Training STGNN ...
STGNN | RMSE=0.075300 | MAE=0.042600 | R2=0.839800

Training GCN-Transformer ...
GCN-Transformer | RMSE=0.069800 | MAE=0.036400 | R2=0.893500

Training GAT-Transformer ...
GAT-Transformer | RMSE=0.066500 | MAE=0.032100 | R2=0.957800

Training SWO-STGCN [Proposed] ...
SWO-STGCN [Proposed] | RMSE=0.061800 | MAE